In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import sys

!{sys.executable} -m pip install -q kagglehub evaluate wandb tensorboard bitsandbytes accelerate peft transformers torch scikit-learn tqdm matplotlib seaborn
!pip install --upgrade transformers
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import psutil
import warnings
warnings.filterwarnings('ignore')


print("All libraries installed successfully!")


# Dataset Download and processing

In [ ]:
import kagglehub

path = kagglehub.dataset_download("peiyuanliu2001/mmlu-dataset")

print("Dataset path:", path)

In [ ]:
import pandas as pd
from datasets import Dataset

# Load the dataset
df = pd.read_csv(f"{path}/train.csv")

print("Dataset Overview:")
print(f"  Total samples: {len(df):,}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\nAnswer distribution:")
print(df['answer'].value_counts().to_dict())

print("\nSample questions:")
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

def format_mmlu_prompt(row):
    """Format MMLU questions for EVALUATION (no answer)."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer:"""
    return prompt

def format_mmlu_prompt_with_answer(row):
    """Format MMLU questions for TRAINING (includes answer)."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer: {row['answer']}"""
    return prompt

stratify_labels = df['subject'] if 'subject' in df.columns else None
train_indices, eval_indices = train_test_split(
    df.index,
    test_size=0.3,
    random_state=42,
    stratify=stratify_labels if stratify_labels is not None else None,
)

df['split'] = 'train'
df.loc[eval_indices, 'split'] = 'eval'
train_subset_for_examples = df.loc[train_indices].copy()

df['formatted_prompt'] = df.apply(format_mmlu_prompt, axis=1)  # For evaluation
df['formatted_prompt_with_answer'] = df.apply(format_mmlu_prompt_with_answer, axis=1)  # For training

answer_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['answer_idx'] = df['answer'].map(answer_to_idx)

print("\nSample formatted prompt (EVALUATION - no answer):")
sample_prompt = df['formatted_prompt'].iloc[0]
print(sample_prompt[:500] + "..." if len(sample_prompt) > 500 else sample_prompt)

print(f"\nSample formatted prompt (TRAINING - with answer):")
sample_prompt_train = df['formatted_prompt_with_answer'].iloc[0]
print(sample_prompt_train[:550] + "..." if len(sample_prompt_train) > 550 else sample_prompt_train)

print(f"\nExpected answer: {df['answer'].iloc[0]}")

train_df = df[df['split'] == 'train'].copy()
eval_df = df[df['split'] == 'eval'].copy()

print(f"\nSplit summary:")
print(f"  Train samples: {len(train_df):,}")
print(f"  Eval samples:  {len(eval_df):,}")


In [ ]:
from datasets import Dataset, DatasetDict


MAX_LENGTH = 512 
# Determine columns to include
columns_to_use = ['prompt', 'formatted_prompt', 'formatted_prompt_with_answer', 'answer', 'answer_idx']
if 'subject' in train_df.columns:
    columns_to_use.insert(0, 'subject')
if 'A' in train_df.columns:
    columns_to_use.extend(['A', 'B', 'C', 'D'])

train_dataset_raw = Dataset.from_pandas(
    train_df[columns_to_use],
    preserve_index=False,
)
eval_dataset_raw = Dataset.from_pandas(
    eval_df[columns_to_use],
    preserve_index=False,
)

dataset = DatasetDict({
    'train': train_dataset_raw,
    'test': eval_dataset_raw,
})
eval_dataset = eval_dataset_raw

print("Dataset prepared:")
print(f"  Training samples: {len(train_dataset_raw):,}")
print(f"  Evaluation samples: {len(eval_dataset_raw):,}")
print(f"  Columns: {eval_dataset_raw.column_names}")


# Model loading (8-bit quantized)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the model path
model_id = "mistralai/Mistral-7B-v0.1"


bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_compute_dtype=torch.bfloat16,  
    bnb_8bit_quant_type="nf4", 
    llm_int8_enable_fp32_cpu_offload=True 
)

device_map = {"": "cuda:0"}

print("Loading model with 8-bit quantization (QLoRA)...")


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True
)

base_model = model

Tokenizer


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Set pad_token to eos_token (required for padding during training)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Get token IDs for answer choices
ANSWER_TOKENS = {
    'A': tokenizer.encode('A', add_special_tokens=False)[0],
    'B': tokenizer.encode('B', add_special_tokens=False)[0],
    'C': tokenizer.encode('C', add_special_tokens=False)[0],
    'D': tokenizer.encode('D', add_special_tokens=False)[0],
}

print("Answer token IDs:")
for letter, token_id in ANSWER_TOKENS.items():
    decoded = tokenizer.decode([token_id])
    print(f"  {letter}: {token_id} → '{decoded}'")

# Also create reverse mapping
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}



# Evaluation functions

In [ ]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import time

def compute_ece(confidences, predictions, labels, n_bins=10):
    """
    Compute Expected Calibration Error (ECE).
    Measures how well model confidence aligns with actual accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()

        if prop_in_bin > 0:
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

def evaluate_mmlu_comprehensive(model, tokenizer, eval_dataset, answer_tokens,
                                 device="cuda", max_samples=None, show_progress=True):
    """
    Comprehensive MMLU evaluation with all Priority 1-3 metrics.

    Returns dict with:
    - accuracy: Overall accuracy
    - top2_accuracy: Top-2 accuracy
    - confidences: Model confidence scores
    - predictions: Predicted answers
    - true_labels: Ground truth answers
    - confusion_matrix: Confusion matrix
    - ece: Expected Calibration Error
    - throughput: Samples per second
    - avg_latency: Average latency per sample
    """
    model.eval()

    correct = 0
    top2_correct = 0
    total = 0

    all_confidences = []
    all_predictions = []
    all_true_labels = []

    # Limit samples if specified
    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    # Get token IDs as tensor
    answer_token_ids = torch.tensor([answer_tokens['A'], answer_tokens['B'],
                                      answer_tokens['C'], answer_tokens['D']])

    # Start timing
    start_time = time.time()

    iterator = tqdm(samples, desc="Evaluating", disable=not show_progress)

    with torch.no_grad():
        for example in iterator:
            prompt = example['formatted_prompt']
            true_answer = example['answer']
            true_idx = answer_to_idx[true_answer]

            # Tokenize and get logits
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, :]

            # Get probabilities for A, B, C, D
            answer_logits = last_logits[answer_token_ids]
            answer_probs = torch.softmax(answer_logits, dim=0)

            # Top-1 prediction
            pred_idx = answer_probs.argmax().item()
            pred_answer = idx_to_letter[pred_idx]
            confidence = answer_probs[pred_idx].item()

            # Top-2 prediction
            top2_indices = answer_probs.topk(2).indices.tolist()

            # Store results
            all_confidences.append(confidence)
            all_predictions.append(pred_idx)
            all_true_labels.append(true_idx)

            if pred_answer == true_answer:
                correct += 1
            if true_idx in top2_indices:
                top2_correct += 1
            total += 1

    # End timing
    end_time = time.time()
    duration = end_time - start_time

    # Convert to numpy arrays
    confidences = np.array(all_confidences)
    predictions = np.array(all_predictions)
    true_labels = np.array(all_true_labels)

    # Compute metrics
    accuracy = correct / total if total > 0 else 0.0
    top2_accuracy = top2_correct / total if total > 0 else 0.0
    ece = compute_ece(confidences, predictions, true_labels)
    conf_matrix = confusion_matrix(true_labels, predictions, labels=[0, 1, 2, 3])

    # Performance metrics
    throughput = total / duration if duration > 0 else 0.0
    avg_latency = duration / total if total > 0 else 0.0

    return {
        'accuracy': accuracy,
        'top2_accuracy': top2_accuracy,
        'correct': correct,
        'total': total,
        'ece': ece,
        'confidences': confidences,
        'predictions': predictions,
        'true_labels': true_labels,
        'confusion_matrix': conf_matrix,
        'throughput': throughput,
        'avg_latency': avg_latency,
    }

import time
import gc

def compute_model_flops(model, seq_length=512):
    """
    Estimate FLOPs per forward pass for transformer model.

    Formula: FLOPs ≈ 2 * active_params * seq_length
    For MoE: multiply by sparsity factor (active_experts / total_experts)
    """
    config = model.config
    n_layers = config.num_hidden_layers
    d_model = config.hidden_size
    intermediate_size = config.intermediate_size
    vocab_size = config.vocab_size

    # Attention FLOPs per layer: 4 * seq_len * d_model^2
    attention_flops = 4 * seq_length * d_model * d_model * n_layers

    # FFN FLOPs per layer: 2 * seq_len * d_model * intermediate_size * 2
    ffn_flops = 8 * seq_length * d_model * intermediate_size * n_layers

    total_flops = attention_flops + ffn_flops

    # Check if MoE model
    is_moe = False
    sparsity_factor = 1.0
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                sparsity_factor = num_experts_per_tok / num_experts
                break
    except:
        pass

    if is_moe:
        # Apply sparsity to FFN only (attention is unchanged)
        total_flops = attention_flops + (ffn_flops * sparsity_factor)

    return total_flops


def compute_throughput_metrics(model, tokenizer, eval_dataset, max_samples=100):
    """
    Measure throughput and latency metrics.

    Returns:
        - tokens_per_second: Throughput in tokens/sec
        - ms_per_token: Latency in ms/token
        - samples_per_second: Sample throughput
        - total_time: Total evaluation time
    """
    model.eval()

    # Take subset
    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    total_tokens = 0
    sample_times = []

    with torch.no_grad():
        for example in samples:
            prompt = example['formatted_prompt']

            # Tokenize
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                             max_length=MAX_LENGTH).to("cuda")
            num_tokens = inputs['input_ids'].shape[1]
            total_tokens += num_tokens

            # Time inference
            start = time.time()
            _ = model(**inputs)
            sample_time = time.time() - start
            sample_times.append(sample_time)

    total_time = sum(sample_times)
    tokens_per_second = total_tokens / total_time if total_time > 0 else 0
    ms_per_token = (total_time / total_tokens * 1000) if total_tokens > 0 else 0
    samples_per_second = len(samples) / total_time if total_time > 0 else 0

    return {
        'tokens_per_second': tokens_per_second,
        'ms_per_token': ms_per_token,
        'samples_per_second': samples_per_second,
        'total_time': total_time,
    }


def compute_parameter_efficiency(model, num_experts_per_tok=1):
    """
    Calculate parameter efficiency metrics.

    Returns:
        - total_params: Total model parameters
        - active_params: Parameters used per forward pass
        - trainable_params: Parameters being trained
        - sparsity_ratio: Fraction of params used (active/total)
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Check if MoE
    is_moe = False
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                break
    except:
        pass

    if is_moe:
        # Calculate active params correctly for MoE
        # Active = attention params + (k/n * expert params)
        # where k = experts per token, n = total experts
        
        # Get total expert parameters across all layers
        total_expert_params = 0
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                # Sum all expert FFN parameters in this layer
                if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, list):
                    for expert_idx in range(num_experts):
                        total_expert_params += sum(p.numel() for p in layer.mlp.gate_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.up_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.down_proj[expert_idx].parameters())
        
        # Active expert params = (k/n) * total expert params
        sparsity = num_experts_per_tok / num_experts
        active_expert_params = int(total_expert_params * sparsity)
        
        # Non-expert params (attention, embeddings, etc)
        non_expert_params = total_params - total_expert_params
        
        # Total active = all non-expert params + active expert params
        active_params = non_expert_params + active_expert_params
    else:
        active_params = total_params

    sparsity_ratio = active_params / total_params if total_params > 0 else 1.0

    return {
        'total_params': total_params,
        'active_params': active_params,
        'trainable_params': trainable_params,
        'sparsity_ratio': sparsity_ratio,
    }


def compute_memory_metrics(model):
    """
    Compute memory usage statistics.

    Returns:
        - model_size_mb: Model size in memory
        - gpu_memory_allocated_gb: GPU memory allocated
        - gpu_memory_reserved_gb: GPU memory reserved
    """
    # Model size
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / 1024 / 1024

    # GPU memory
    if torch.cuda.is_available():
        gpu_memory_allocated_gb = torch.cuda.memory_allocated() / 1024 / 1024 / 1024
        gpu_memory_reserved_gb = torch.cuda.memory_reserved() / 1024 / 1024 / 1024
    else:
        gpu_memory_allocated_gb = 0
        gpu_memory_reserved_gb = 0

    return {
        'model_size_mb': model_size_mb,
        'gpu_memory_allocated_gb': gpu_memory_allocated_gb,
        'gpu_memory_reserved_gb': gpu_memory_reserved_gb,
    }

print("evaluation functions defined")

In [ ]:
def collect_router_statistics(model, eval_dataset, tokenizer, answer_tokens,
                               max_samples=500, device="cuda"):
    """
    Collect router statistics from MoE model during inference.

    Args:
        model: MoE model
        eval_dataset: Dataset to evaluate on
        tokenizer: Tokenizer
        answer_tokens: Answer token IDs
        max_samples: Number of samples to analyze
        device: Device to run on

    Returns:
        Dict with router statistics
    """
    model.eval()

    # Detect MoE config
    first_moe_layer = None
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'num_experts'):
            first_moe_layer = layer.mlp
            break

    if first_moe_layer:
        num_experts = first_moe_layer.num_experts
        num_experts_per_tok = first_moe_layer.num_experts_per_tok
    else:
        num_experts = globals().get('NUM_EXPERTS', 8)
        num_experts_per_tok = globals().get('NUM_EXPERTS_PER_TOK', 2)

    # Enable router logits collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'forward'):
            layer.mlp._collect_router_logits = True

    # Storage for router statistics
    router_stats = {
        'expert_selections': defaultdict(lambda: np.zeros(num_experts)),
        'expert_confidence': defaultdict(list),
        'per_layer_selection': [np.zeros(num_experts) for _ in range(len(model.model.layers))],
        'per_subject_routing': defaultdict(lambda: defaultdict(lambda: np.zeros(num_experts))),
    }

    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    print(f"Collecting router statistics from {len(samples)} samples (Experts: {num_experts}, k: {num_experts_per_tok})...")

    with torch.no_grad():
        for example in tqdm(samples, desc="Collecting router stats"):
            prompt = example['formatted_prompt']
            subject = example.get('subject', 'default')

            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            # Forward pass
            outputs = model(**inputs)

            # Collect router information from each layer
            for layer_idx, layer in enumerate(model.model.layers):
                if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
                    router_probs = layer.mlp._last_router_probs.cpu().numpy()

                    # Average across all tokens in sequence
                    avg_probs = router_probs.mean(axis=0)

                    # Track which experts are selected (top-k)
                    top_experts = np.argsort(avg_probs)[-num_experts_per_tok:]

                    # Update statistics
                    router_stats['per_layer_selection'][layer_idx] += avg_probs

                    for expert_idx in top_experts:
                        router_stats['expert_selections']['overall'][expert_idx] += 1
                        router_stats['per_subject_routing'][subject][layer_idx][expert_idx] += 1

                    # Track confidence (probability of top expert)
                    router_stats['expert_confidence'][layer_idx].append(avg_probs.max())

    # Normalize statistics
    for layer_idx in range(len(router_stats['per_layer_selection'])):
        total = router_stats['per_layer_selection'][layer_idx].sum()
        if total > 0:
            router_stats['per_layer_selection'][layer_idx] /= total

    # Disable router collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_collect_router_logits'):
            layer.mlp._collect_router_logits = False

    print("Router statistics collected!")
    return router_stats


def visualize_router_statistics(router_stats, title="MoE Router Analysis"):
    """
    Create comprehensive visualizations of router behavior.

    Args:
        router_stats: Router statistics from collect_router_statistics
        title: Title for the analysis
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Dynamically detect num_experts from data
    expert_usage = router_stats['expert_selections']['overall']
    num_experts = len(expert_usage)

    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1. Expert utilization across all layers
    ax1 = fig.add_subplot(gs[0, :2])
    if expert_usage.sum() > 0:
        expert_usage_norm = expert_usage / expert_usage.sum()
    else:
        expert_usage_norm = expert_usage

    bars = ax1.bar(range(num_experts), expert_usage_norm, color='steelblue', alpha=0.7)
    ax1.axhline(1/num_experts, color='red', linestyle='--', label='Uniform distribution')
    ax1.set_xlabel('Expert ID', fontsize=12)
    ax1.set_ylabel('Selection Frequency', fontsize=12)
    ax1.set_title('Overall Expert Utilization (All Layers)', fontsize=14, fontweight='bold')
    ax1.set_xticks(range(num_experts))
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Add percentage labels on bars
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height*100:.1f}%', ha='center', va='bottom', fontsize=9)

    # 2. Expert utilization heatmap across layers
    ax2 = fig.add_subplot(gs[0, 2])
    layer_expert_matrix = np.array(router_stats['per_layer_selection'])
    sns.heatmap(layer_expert_matrix.T, cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Selection Prob'})
    ax2.set_xlabel('Layer', fontsize=10)
    ax2.set_ylabel('Expert ID', fontsize=10)
    ax2.set_title('Expert Selection Heatmap\n(Layer vs Expert)', fontsize=12, fontweight='bold')

    # 3. Load balance score per layer
    ax3 = fig.add_subplot(gs[1, 0])
    load_balance_scores = []
    for layer_probs in router_stats['per_layer_selection']:
        # Compute entropy as measure of load balance
        if layer_probs.sum() > 0:
            layer_probs_norm = layer_probs / layer_probs.sum()
            entropy = -np.sum(layer_probs_norm * np.log(layer_probs_norm + 1e-10))
            normalized_entropy = entropy / np.log(num_experts)
        else:
            normalized_entropy = 0
        load_balance_scores.append(normalized_entropy)

    ax3.plot(load_balance_scores, marker='o', linewidth=2, markersize=4)
    ax3.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect balance')
    ax3.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='50% balance')
    ax3.set_xlabel('Layer', fontsize=10)
    ax3.set_ylabel('Load Balance Score', fontsize=10)
    ax3.set_title('Load Balancing Across Layers\n(1.0 = perfect balance)', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)
    ax3.set_ylim([0, 1.1])

    # 4. Router confidence distribution
    ax4 = fig.add_subplot(gs[1, 1])
    all_confidences = []
    for layer_confs in router_stats['expert_confidence'].values():
        all_confidences.extend(layer_confs)

    ax4.hist(all_confidences, bins=50, color='purple', alpha=0.7, edgecolor='black')
    ax4.axvline(np.mean(all_confidences), color='red', linestyle='--',
                linewidth=2, label=f"Mean: {np.mean(all_confidences):.3f}")
    ax4.set_xlabel('Router Confidence (Max Prob)', fontsize=10)
    ax4.set_ylabel('Frequency', fontsize=10)
    ax4.set_title('Distribution of Router Confidence', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)

    # 5. Expert specialization: variance across layers
    ax5 = fig.add_subplot(gs[1, 2])
    expert_variances = []
    for expert_id in range(num_experts):
        expert_across_layers = layer_expert_matrix[:, expert_id]
        variance = np.var(expert_across_layers)
        expert_variances.append(variance)

    ax5.bar(range(num_experts), expert_variances, color='teal', alpha=0.7)
    ax5.set_xlabel('Expert ID', fontsize=10)
    ax5.set_ylabel('Variance Across Layers', fontsize=10)
    ax5.set_title('Expert Specialization\n(Higher = more layer-specific)', fontsize=12, fontweight='bold')
    ax5.set_xticks(range(num_experts))
    ax5.grid(axis='y', alpha=0.3)

    # 6. Per-layer confidence box plots
    ax6 = fig.add_subplot(gs[2, :])
    layer_confidence_data = [router_stats['expert_confidence'][i]
                             for i in range(len(router_stats['expert_confidence']))]
    bp = ax6.boxplot(layer_confidence_data, patch_artist=True, showmeans=True)

    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    ax6.set_xlabel('Layer', fontsize=12)
    ax6.set_ylabel('Router Confidence', fontsize=12)
    ax6.set_title('Router Confidence Distribution Across Layers', fontsize=14, fontweight='bold')
    ax6.grid(axis='y', alpha=0.3)

    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('router_visualization.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to 'router_visualization.png'")
    plt.show()

    # Print statistics
  
    print("ROUTER STATISTICS SUMMARY")
 

    print(f"Average load balance score: {np.mean(load_balance_scores):.4f}")
    print(f"Average router confidence: {np.mean(all_confidences):.4f}")
    print(f"Std dev of expert usage: {np.std(expert_usage_norm):.4f}")

    # Check for router collapse
    max_expert_usage = expert_usage_norm.max()
    if max_expert_usage > 0.3:
        print(f"\nWARNING: Potential router collapse detected!")
        print(f"   Expert {expert_usage_norm.argmax()} is selected {max_expert_usage*100:.1f}% of the time")
    else:
        print(f"\nNo router collapse detected (max usage: {max_expert_usage*100:.1f}%)")


In [ ]:
from transformers import TrainerCallback
from torch.utils.data import DataLoader
import gc

class ComprehensiveEvalCallback(TrainerCallback):
    """
    Custom callback that evaluates and logs all Priority 1-3 metrics every N steps.
    """

    def __init__(self, eval_dataset_for_accuracy, eval_tokenized_for_loss,
                 tokenizer, answer_tokens, eval_steps=50,
                 accuracy_samples=1000, device="cuda"):
        super().__init__()
        self.eval_dataset_for_accuracy = eval_dataset_for_accuracy
        self.eval_tokenized_for_loss = eval_tokenized_for_loss
        self.tokenizer = tokenizer
        self.answer_tokens = answer_tokens
        self.eval_steps = eval_steps
        self.accuracy_samples = accuracy_samples
        self.device = device

        self.metrics_history = []
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

        print(f"Evaluating every {self.eval_steps} steps")

        return control

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.eval_steps == 0 and state.global_step > 0:
            self._evaluate_and_log(args, state, model)
        return control

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        print(f"END OF EPOCH {state.epoch:.0f}")

        self._evaluate_and_log(args, state, model, is_epoch_end=True)
        return control

    def _evaluate_and_log(self, args, state, model, is_epoch_end=False):
        """Compute and log all metrics."""

        # Get training loss from state
        train_loss = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'loss' in log:
                    train_loss = log['loss']
                    break

        # Get learning rate
        learning_rate = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'learning_rate' in log:
                    learning_rate = log['learning_rate']
                    break

        # Compute evaluation loss and throughput
        eval_loss, eval_throughput, eval_latency = self._compute_eval_loss(model)

        # Compute perplexity
        perplexity = torch.exp(torch.tensor(eval_loss)).item()

        # Comprehensive MMLU evaluation
        mmlu_results = evaluate_mmlu_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset_for_accuracy,
            answer_tokens=self.answer_tokens,
            device=self.device,
            max_samples=self.accuracy_samples,
            show_progress=False
        )

        # GPU memory
        peak_gpu_memory = self._get_peak_gpu_memory()

        # Calculate elapsed time
        elapsed = time.time() - self.start_time

        # Gradient norm (if available)
        grad_norm = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'grad_norm' in log:
                    grad_norm = log['grad_norm']
                    break

        # Store metrics
        metrics = {
            'step': state.global_step,
            'epoch': state.epoch,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            'perplexity': perplexity,
            'learning_rate': learning_rate,
            'grad_norm': grad_norm,
            # MMLU metrics
            'mmlu_accuracy': mmlu_results['accuracy'],
            'mmlu_top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu_ece': mmlu_results['ece'],
            'mmlu_correct': mmlu_results['correct'],
            'mmlu_total': mmlu_results['total'],
            # Performance metrics
            'throughput': mmlu_results['throughput'],
            'avg_latency': mmlu_results['avg_latency'],
            'eval_throughput': eval_throughput,
            'eval_latency': eval_latency,
            'peak_gpu_memory_gb': peak_gpu_memory,
            'elapsed_time': elapsed,
        }
        self.metrics_history.append(metrics)

        # Print metrics
        step_info = f"Epoch {state.epoch:.2f}" if is_epoch_end else f"Step {state.global_step}"

        print(f"EVALUATION at {step_info}")
        print(f"  Train Loss:      {train_loss:.4f}" if train_loss else "  Train Loss:      N/A")
        print(f"  Eval Loss:       {eval_loss:.4f}")
        print(f"  MMLU Accuracy:   {mmlu_results['accuracy']:.4f} ({mmlu_results['correct']}/{mmlu_results['total']})")
        print(f"  Perplexity:      {perplexity:.2f}")
        print(f"  Throughput:      {mmlu_results['throughput']:.2f} samples/sec")
        print(f"  Avg Latency:     {mmlu_results['avg_latency']:.4f} sec/sample")
        print(f"  Peak GPU Memory: {peak_gpu_memory:.2f} GB")
        print(f"  ECE (Calibration): {mmlu_results['ece']:.4f}")
        print(f"  Top-2 Accuracy:  {mmlu_results['top2_accuracy']:.4f}")
        if learning_rate:
            print(f"  Learning Rate:   {learning_rate:.2e}")
        if grad_norm:
            print(f"  Gradient Norm:   {grad_norm:.4f}")

        print(f" Elapsed Time:    {elapsed/60:.1f} min")


        # Clean up
        torch.cuda.empty_cache()
        gc.collect()
        model.train()

    def _compute_eval_loss(self, model, num_batches=50):
        """Compute average loss on evaluation set."""
        model.eval()
        total_loss = 0
        num_samples = 0

        eval_dataloader = DataLoader(
            self.eval_tokenized_for_loss,
            batch_size=1,
            shuffle=False
        )

        start_time = time.time()

        with torch.no_grad():
            for i, batch in enumerate(eval_dataloader):
                if i >= num_batches:
                    break

                # FIX: Convert lists to tensors and ensure 2D shape [batch, seq_len]
                processed_batch = {}
                for k, v in batch.items():
                    if isinstance(v, list):
                        v = torch.tensor(v)
                    # Ensure tensor is 2D (add batch dimension if needed)
                    if v.dim() == 1:
                        v = v.unsqueeze(0)
                    processed_batch[k] = v.to(self.device)
                
                # FIX: Add labels if not present (needed for loss computation)
                if 'labels' not in processed_batch and 'input_ids' in processed_batch:
                    processed_batch['labels'] = processed_batch['input_ids'].clone()
                
                outputs = model(**processed_batch)
                
                # Skip if loss is still None (shouldn't happen now)
                if outputs.loss is None:
                    continue
                    
                total_loss += outputs.loss.item()
                num_samples += 1

        end_time = time.time()
        duration = end_time - start_time

        avg_loss = total_loss / num_samples if num_samples > 0 else 0
        throughput = num_samples / duration if duration > 0 else 0.0
        latency = duration / num_samples if num_samples > 0 else 0.0

        return avg_loss, throughput, latency

    def _get_peak_gpu_memory(self):
        """Get peak GPU memory and reset stats."""
        if torch.cuda.is_available():
            max_mem = torch.cuda.max_memory_allocated() / (1024**3)
            torch.cuda.reset_peak_memory_stats()
            return max_mem
        return 0.0

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time

        print(f"TRAINING COMPLETED in {elapsed/60:.1f} minutes")

        # Print summary table
        if self.metrics_history:
            print("TRAINING METRICS SUMMARY:")
            print(f"{'Step':<8} {'Epoch':<7} {'Train':<8} {'Eval':<8} {'Perp':<7} {'MMLU':<8} {'Top-2':<8} {'ECE':<7} {'Latency':<9}")
            print(f"{'':8} {'':7} {'Loss':<8} {'Loss':<8} {'':7} {'Acc':<8} {'Acc':<8} {'':7} {'(sec)':<9}")
            print("-" * 80)
            for m in self.metrics_history:
                train_loss_str = f"{m['train_loss']:.4f}" if m['train_loss'] else "N/A"
                print(f"{m['step']:<8} {m['epoch']:<7.2f} {train_loss_str:<8} {m['eval_loss']:<8.4f} "
                      f"{m['perplexity']:<7.2f} {m['mmlu_accuracy']:<8.4f} {m['mmlu_top2_accuracy']:<8.4f} "
                      f"{m['mmlu_ece']:<7.4f} {m['avg_latency']:<9.4f}")

        return control


print(" Comprehensive evaluation callback with batched eval defined")

# Dense Model Baseline Evaluation

In [ ]:

print("BASELINE EVALUATION")


# 1. MMLU Accuracy
baseline_results = evaluate_mmlu_comprehensive(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\nComputing FLOPs...")
baseline_flops = compute_model_flops(base_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print("Measuring throughput...")
baseline_throughput = compute_throughput_metrics(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print("Analyzing parameter efficiency...")
baseline_params = compute_parameter_efficiency(
    model=base_model,
    num_experts_per_tok=1  # Dense model
)

# 5. Memory metrics
print("Collecting memory metrics...")
baseline_memory = compute_memory_metrics(base_model)

# Combine all metrics
baseline_comprehensive = {
    **baseline_results,
    'flops': baseline_flops,
    **baseline_throughput,
    **baseline_params,
    **baseline_memory,
}

# Display comprehensive results

print("COMPREHENSIVE BASELINE METRICS")


print("Accuracy Metrics:")
print(f"  MMLU Accuracy: {baseline_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {baseline_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {baseline_comprehensive['ece']:.4f}")

print("\n Computational Efficiency:")
print(f"  FLOPs per forward pass: {baseline_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {baseline_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {baseline_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {baseline_comprehensive['samples_per_second']:.2f}")

print("\n Parameter Efficiency:")
print(f"  Total parameters: {baseline_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {baseline_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {baseline_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {baseline_comprehensive['sparsity_ratio']:.2%}")

print("\n Memory Usage:")
print(f"  Model size: {baseline_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {baseline_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# Save comprehensive results
import json
import os
os.makedirs("results", exist_ok=True)
with open("results/baseline_comprehensive.json", 'w') as f:
    json.dump({k: v for k, v in baseline_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)

# Log to wandb with detailed metrics
try:
    if wandb.run is not None:
        wandb.log({
            'baseline/accuracy': baseline_comprehensive['accuracy'],
            'baseline/flops_billions': baseline_comprehensive['flops']/1e9,
            'baseline/tokens_per_second': baseline_comprehensive['tokens_per_second'],
            'baseline/ms_per_token': baseline_comprehensive['ms_per_token'],
            'baseline/active_params_billions': baseline_comprehensive['active_params']/1e9,
            'baseline/gpu_memory_gb': baseline_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

# Store for later comparison
baseline_accuracy = baseline_comprehensive['accuracy']

print("\n Comprehensive baseline evaluation complete!")

# Knowledge distillation setup

In [ ]:
# Teacher = dense baseline
teacher_model = base_model.eval()
teacher_baseline_results = baseline_comprehensive.copy()
print(f" Teacher model set (dense). Teacher accuracy: {teacher_baseline_results['accuracy']:.4f}")

# KD config 1: Output-only distillation (stable)
KD_CONFIG_STANDARD = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': False,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.001,
    'name': 'Standard KD',
}

# KD config 2: Output + light router constraints (MoE-stable)
KD_CONFIG_ROUTER_STABLE = {
    'kd_alpha': 0.6,
    'temperature': 5.0,
    'routing_kd_weight': 0.1,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': True,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.01,
    'name': 'Router-Stable KD',
}

print("KD configs ready (Standard, Router-Stable)")

# Baseline MoE implementation

In [ ]:
# ============================================================================
# DENSE MODEL KNOWLEDGE DISTILLATION TRAINER
# ============================================================================

from transformers import Trainer
import torch.nn.functional as F

class DenseKDTrainer(Trainer):
    """
    Knowledge Distillation Trainer for Dense Models.
    Loss: L_total = (1-α)*L_NTP + α*L_KD
    
    Where:
    - L_NTP: Next Token Prediction loss (standard supervised learning)
    - L_KD: Knowledge Distillation loss (KL divergence between teacher and student logits)
    - α: Weighting factor for KD loss (typically 0.5)
    """
    def __init__(self, teacher_model=None, kd_config=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        
        if self.teacher_model is not None:
            self.teacher_model.eval()
            for param in self.teacher_model.parameters():
                param.requires_grad = False
            print(f" Teacher model loaded (frozen, KD alpha={self.kd_alpha}, T={self.temperature})")

    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Compute loss with Knowledge Distillation.
        """
        # Forward pass through student model
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        # Initialize KD loss
        kd_loss = torch.tensor(0.0, device=device)

        # Compute KD loss if teacher model is available
        if self.teacher_model is not None and self.kd_alpha > 0:
            # Move inputs to teacher's device
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            
            # Forward pass through teacher model (no gradients)
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            # Apply temperature scaling
            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            # KL divergence loss
            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

        # Combined loss
        total_loss = (1 - self.kd_alpha) * ntp_loss + self.kd_alpha * kd_loss

        return (total_loss, outputs) if return_outputs else total_loss

print(" DenseKDTrainer class defined")

In [ ]:
import torch
import torch.nn as nn

class MoELayer(nn.Module):
    """
    MoE Layer with:
    - Proper router initialization for top-2 routing  
    - VECTORIZED output combination (FAST - no Python loops)
    - No expert noise - all experts start identical
    - FIXED: Uses regular nn.Linear for bf16 experts (no broken quantization)
    """
    
    def __init__(self, hidden_size, intermediate_size, num_experts=8,
                 num_experts_per_tok=2, router_jitter_noise=0.0,
                 router_aux_loss_coef=0.001, bnb_config=None, device="cuda",
                 init_on_cpu=True, dtype=torch.bfloat16, enable_cpu_offload=True):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.router_jitter_noise = router_jitter_noise
        self.router_aux_loss_coef = router_aux_loss_coef
        self.bnb_config = bnb_config
        self.device = device
        self.dtype = dtype
        self.enable_cpu_offload = False  # Disabled - causes issues
        
        # ========================================================================
        # FIXED: Always use nn.Linear for experts (bf16)
        # Quantization should happen AFTER weight copying, not during layer creation
        # ========================================================================
        LinearClass = nn.Linear
        self.compute_dtype = dtype
        init_device = 'cpu' if init_on_cpu else device
        linear_kwargs = {'device': init_device, 'dtype': dtype}
        
        # Initialize experts with regular nn.Linear (will copy weights later)
        self.gate_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.up_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.down_proj = nn.ModuleList([
            LinearClass(intermediate_size, hidden_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        
        # Router initialization - CRITICAL for preserving pretrained accuracy
        # When all experts are identical, we need consistent routing to get same output
        # 
        # FIXED: Use bias=True to add constant offsets that don't depend on input values
        # The weight*input product varies with input, but bias is constant
        self.router = nn.Linear(hidden_size, num_experts, bias=True, device=device, dtype=dtype)
        with torch.no_grad():
            # Zero out weights so logits don't depend on input content
            self.router.weight.zero_()
            # Use BIAS to control which experts are selected (constant regardless of input)
            # Expert 0 gets highest bias - will always be in top-2
            self.router.bias[0] = 10.0
            # Expert 1 gets second highest - will always be in top-2  
            self.router.bias[1] = 9.0
            # All other experts get negative bias - will never be selected
            if num_experts > 2:
                self.router.bias[2:] = -10.0
        
        self._last_router_probs = None
        self._collect_router_logits = False
        self._experts_on_gpu = set()
    
    def forward(self, hidden_states):
        """
        VECTORIZED forward pass (FAST - no Python loops).
        """
        original_dtype = hidden_states.dtype
        hidden_states_reshaped = hidden_states.view(-1, hidden_states.size(-1))  # [B*S, H]
        
        # Router computation
        router_input = hidden_states_reshaped.to(self.router.weight.dtype)
        router_logits = self.router(router_input)  # [B*S, num_experts]
        
        if self.training and self.router_jitter_noise > 0:
            router_logits = router_logits + torch.normal(
                0, self.router_jitter_noise, size=router_logits.shape, device=router_logits.device
            )
        
        router_probs = torch.softmax(router_logits, dim=-1)  # [B*S, num_experts]
        top_k_probs, top_k_indices = torch.topk(router_probs, self.num_experts_per_tok, dim=-1)  # [B*S, k]

        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        if self.training:
            self._last_router_probs = router_probs.view(hidden_states.size(0), hidden_states.size(1), -1)
        
        # VECTORIZED output combination (FAST)
        output = torch.zeros_like(hidden_states_reshaped)
        
        for expert_idx in range(self.num_experts):
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)  # [B*S]
            if expert_mask.sum() == 0:
                continue
            
            # VECTORIZED: Compute weights for all tokens at once
            expert_selected_mask = (top_k_indices == expert_idx)  # [B*S, k]
            expert_weights = (top_k_probs * expert_selected_mask.float()).sum(dim=-1)  # [B*S]
            
            # Apply expert FFN
            expert_hidden = hidden_states_reshaped[expert_mask]
            
            if self.compute_dtype is not None and expert_hidden.dtype != self.compute_dtype:
                expert_hidden = expert_hidden.to(dtype=self.compute_dtype)
            
            gate_out = torch.nn.functional.silu(self.gate_proj[expert_idx](expert_hidden))
            up_out = self.up_proj[expert_idx](expert_hidden)
            expert_out = gate_out * up_out
            expert_out = self.down_proj[expert_idx](expert_out)
            expert_out = expert_out.to(dtype=output.dtype)
            
            # VECTORIZED: Weighted combination
            output[expert_mask] += expert_weights[expert_mask].unsqueeze(-1) * expert_out
        
        output = output.view_as(hidden_states)
        return output.to(original_dtype)
    
    def compute_auxiliary_loss(self):
        """
        Compute load balancing auxiliary loss.
        FIXED: Returns raw aux_loss without coefficient - coefficient is applied in MoETrainer.
        """
        if self._last_router_probs is None:
            return torch.tensor(0.0, device=self.router.weight.device)
        expert_freq = self._last_router_probs.mean(dim=[0, 1])
        router_confidence = self._last_router_probs.mean(dim=[0, 1])
        aux_loss = torch.sum(expert_freq * router_confidence) * self.num_experts
        return aux_loss  # FIXED: Removed "* self.router_aux_loss_coef" - applied in trainer instead

print("MoELayer class defined")
print("Router initialized for top-2 routing")

In [ ]:
# ============================================================================
# FIXED replace_ffn_with_moe - No Double Quantization, No Expert Noise
# ============================================================================
# Fixes:
# 1.  No expert noise - all experts start identical (preserves pretrained knowledge)
# 2.  Minimized quantization errors (proper dequantization)
# 3.  Proper weight extraction and copying
# ============================================================================

def replace_ffn_with_moe(model, num_experts=8, num_experts_per_tok=2,
                         router_jitter_noise=0.0, router_aux_loss_coef=0.001,
                         bnb_config=None, ram_threshold=50.0, use_disk_offload=True,
                         layer_indices=None, half_width=False, enable_cpu_offload=True):
    """
    FIXED: Replace dense FFN layers with MoE layers.
    
    Key improvements:
    - No expert noise - all experts start identical (preserves pretrained knowledge)
    - Minimized quantization errors (Linear4bit handles quantization automatically)
    - Proper weight extraction and copying
    """
    import gc
    import psutil
    import os
    from bitsandbytes.nn import Linear4bit as BnbLinear
    from bitsandbytes.functional import dequantize_4bit
    
    # Set CUDA allocator config
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
    
    config = model.config
    hidden_size = config.hidden_size
    
    # Use half-width experts if specified
    if half_width:
        intermediate_size = config.intermediate_size // 2
        print(f"   Using HALF-WIDTH experts (intermediate_size // 2)")
    else:
        intermediate_size = config.intermediate_size
    
    print(f"Model configuration:")
    print(f"  Hidden size: {hidden_size}")
    print(f"  Original intermediate size: {config.intermediate_size}")
    print(f"  MoE intermediate size: {intermediate_size}")
    print(f"  Number of layers: {config.num_hidden_layers}")
    print(f"  Experts per layer: {num_experts}")
    print(f"  Experts per token: {num_experts_per_tok}")
    
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available!")
    
    device = torch.device("cuda")
    if bnb_config and hasattr(bnb_config, 'bnb_4bit_compute_dtype'):
        compute_dtype = bnb_config.bnb_4bit_compute_dtype
    elif bnb_config and hasattr(bnb_config, 'llm_int8_threshold'):
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.bfloat16
    
    print(f"\n Using GPU for weight processing")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Compute dtype: {compute_dtype}")
    
    # Helper functions
    def check_ram():
        return psutil.virtual_memory().percent
    
    def cleanup(aggressive=False):
        gc.collect()
        torch.cuda.empty_cache()
        if aggressive:
            torch.cuda.synchronize()
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except:
            pass
        gc.collect()
    
    def print_memory_stats(label=""):
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    [{label}] GPU: {allocated:.2f}GB alloc / {reserved:.2f}GB reserved")
    
    def extract_weight(linear_layer, expected_shape, keep_on_cpu=True):
        """
        Extract weight from a layer.
        SIMPLIFIED: Now expects bf16 source model (no quantization).
        """
        expected_numel = torch.Size(expected_shape).numel() if isinstance(expected_shape, (tuple, list)) else expected_shape.numel()
        
        # Simply get the weight data and convert to compute dtype
        weight = linear_layer.weight.data.to(compute_dtype)
        
        # Ensure proper shape
        if weight.shape != expected_shape:
            if weight.numel() == expected_numel:
                weight = weight.reshape(expected_shape)
            elif len(weight.shape) == 2 and len(expected_shape) == 2:
                if weight.shape[0] == expected_shape[1] and weight.shape[1] == expected_shape[0]:
                    weight = weight.t()
                elif weight.numel() == expected_numel:
                    weight = weight.reshape(expected_shape)
        
        return weight.cpu() if keep_on_cpu else weight.cuda()
    
    # Determine layers to process
    total_layers = len(model.model.layers)
    if layer_indices is None:
        target_layers = list(range(total_layers))
    else:
        target_layers = layer_indices
    print(f"\n  Processing {len(target_layers)} layers: {target_layers[:5]}{'...' if len(target_layers) > 5 else ''}\n")
    
    for i in target_layers:
        layer = model.model.layers[i]
        original_mlp = layer.mlp
        
        ram = check_ram()
        print(f"  Layer {i+1}/{total_layers} (RAM: {ram:.1f}%):")
        
        # Step 1: Extract all weights to CPU (memory efficient)
        print(f"    Extracting weights to CPU...", end=" ", flush=True)
        with torch.no_grad():
            # Extract weights and keep on CPU
            gate_w_full = extract_weight(original_mlp.gate_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            up_w_full = extract_weight(original_mlp.up_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            down_w_full = extract_weight(original_mlp.down_proj, (hidden_size, config.intermediate_size), keep_on_cpu=True)
            
            # Slice to target intermediate size if needed
            gate_w = gate_w_full[:intermediate_size, :].clone()
            up_w = up_w_full[:intermediate_size, :].clone()
            down_w = down_w_full[:, :intermediate_size].clone()
            
            # Delete full weights IMMEDIATELY
            del gate_w_full, up_w_full, down_w_full
            gc.collect()
        print("Done")
        
        # Step 2: Delete original MLP to free GPU memory
        print(f"    Deleting original MLP...", end=" ", flush=True)
        del original_mlp
        layer.mlp = None
        cleanup(aggressive=True)
        print("Done")
        
        # Step 3: Create MoE layer
        print(f"    Creating MoE layer...", end=" ", flush=True)
        moe_layer = MoELayer(
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            num_experts=num_experts,
            num_experts_per_tok=num_experts_per_tok,
            router_jitter_noise=router_jitter_noise,
            router_aux_loss_coef=router_aux_loss_coef,
            bnb_config=bnb_config,
            device='cpu' if bnb_config is None else device,
            init_on_cpu=(bnb_config is None),
            dtype=compute_dtype,
            enable_cpu_offload=enable_cpu_offload
        )
        print("Done")
        
        # Step 4: Copy weights to all experts (on CPU) - FIXED: NO NOISE
        print(f"    Copying weights to {num_experts} experts (NO NOISE)...", end=" ", flush=True)
        with torch.no_grad():
            # Validate and fix shapes before copying
            gate_target_shape = moe_layer.gate_proj[0].weight.shape
            up_target_shape = moe_layer.up_proj[0].weight.shape
            down_target_shape = moe_layer.down_proj[0].weight.shape
            
            # Fix gate_w shape if needed
            if gate_w.shape != gate_target_shape:
                if gate_w.numel() == gate_target_shape.numel():
                    gate_w = gate_w.reshape(gate_target_shape)
                else:
                    gate_w = gate_w[:gate_target_shape[0], :gate_target_shape[1]]
            
            # Fix up_w shape if needed
            if up_w.shape != up_target_shape:
                if up_w.numel() == up_target_shape.numel():
                    up_w = up_w.reshape(up_target_shape)
                else:
                    up_w = up_w[:up_target_shape[0], :up_target_shape[1]]
            
            # Fix down_w shape if needed
            if down_w.shape != down_target_shape:
                if down_w.numel() == down_target_shape.numel():
                    down_w = down_w.reshape(down_target_shape)
                elif len(down_w.shape) == 1 or (len(down_w.shape) == 2 and down_w.shape[1] == 1):
                    if down_w.numel() == down_target_shape.numel():
                        down_w = down_w.reshape(down_target_shape)
                    else:
                        transposed_shape = (down_target_shape[1], down_target_shape[0])
                        if down_w.numel() == torch.Size(transposed_shape).numel():
                            down_w = down_w.reshape(transposed_shape).t()
                        else:
                            raise RuntimeError(
                                f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                                f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                            )
                elif down_w.shape[0] == down_target_shape[0] and down_w.shape[1] >= down_target_shape[1]:
                    down_w = down_w[:, :down_target_shape[1]]
                elif down_w.shape[1] == down_target_shape[1] and down_w.shape[0] >= down_target_shape[0]:
                    down_w = down_w[:down_target_shape[0], :]
                elif down_w.shape[0] == down_target_shape[1] and down_w.shape[1] == down_target_shape[0]:
                    down_w = down_w.t()
                else:
                    raise RuntimeError(
                        f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                        f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                    )
            
            # ========================================================================
            # FIXED: Copy weights to ALL experts identically (NO NOISE)
            # ========================================================================
            # All experts start with identical pretrained weights
            # Training will naturally differentiate them through gradient updates
            for idx in range(num_experts):
                moe_layer.gate_proj[idx].weight.copy_(gate_w)
                moe_layer.up_proj[idx].weight.copy_(up_w)
                moe_layer.down_proj[idx].weight.copy_(down_w)
                
                # NO NOISE ADDED - experts are identical initially
                # This preserves pretrained knowledge perfectly
                # The router is initialized to prefer experts 0 & 1 for top-2 routing
                # Training will naturally differentiate all experts
        
        print("Done")
        
        # Delete extracted weights BEFORE moving to GPU
        del gate_w, up_w, down_w
        gc.collect()
        
        # Step 5: Move entire ModuleLists to GPU at once (faster than one-by-one)
        print(f"    Moving to GPU...", end=" ", flush=True)
        moe_layer.gate_proj = moe_layer.gate_proj.to(device)
        moe_layer.up_proj = moe_layer.up_proj.to(device)
        moe_layer.down_proj = moe_layer.down_proj.to(device)
        moe_layer.router = moe_layer.router.to(device)
        print("Done")
        
        # Step 6: Replace and cleanup
        layer.mlp = moe_layer
        cleanup(aggressive=True)
        
        # Progress update every 4 layers
        if (i + 1) % 4 == 0:
            print_memory_stats(f"Layer {i+1}")
    
    cleanup(aggressive=True)
    
    print(f"\n Successfully replaced {len(target_layers)} FFN layers with MoE")
    print(f"  Expert dispatch: Efficient sparse routing (top-{num_experts_per_tok})")
    print(f"  CPU offloading: {'ENABLED' if enable_cpu_offload else 'DISABLED'}")
    print(f"  Params per expert: ~{(intermediate_size * hidden_size * 3) / 1e6:.1f}M")
    print(f"  Active params per token: ~{(intermediate_size * hidden_size * 3 * num_experts_per_tok) / 1e6:.1f}M")
    print(f"   All experts start identical (no noise)")
    print(f"   Router initialized for top-2 routing (experts 0 & 1 preferred)")
    
    # Final stats
    gpu_final = torch.cuda.memory_allocated() / 1e9
    ram_final = check_ram()
    print(f"  Final: GPU {gpu_final:.2f}GB | RAM {ram_final:.1f}%")
    return model

def compute_moe_auxiliary_loss(model, router_aux_loss_coef=0.001):
    """
    Compute auxiliary load balancing loss from all MoE layers.
    L_aux = sum(D_i * P_i) where:
    - D_i = expert frequency (actual utilization rate)
    - P_i = router probability (router confidence)

    Args:
        model: MoE model
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        aux_loss: Auxiliary loss value
    """
    total_aux_loss = torch.tensor(0.0, device=next(model.parameters()).device)

    # Collect router information from all MoE layers
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
            router_probs = layer.mlp._last_router_probs

            # Expert frequency: how often each expert is selected (average probability)
            expert_freq = router_probs.mean(dim=0)  # (num_experts,)

            # Router confidence: average probability assigned to each expert
            router_confidence = router_probs.mean(dim=0)  # (num_experts,)

            # Auxiliary loss: minimize correlation between frequency and confidence
            layer_aux_loss = torch.sum(expert_freq * router_confidence) * layer.mlp.num_experts
            total_aux_loss = total_aux_loss + layer_aux_loss

    return total_aux_loss

def compute_moe_loss(model, outputs, router_aux_loss_coef=0.001):
    """
    Compute total loss for MoE model: L_total = L_NTP + λ * L_aux

    Args:
        model: MoE model
        outputs: Model outputs with loss
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        total_loss: Combined loss
        ntp_loss: Next token prediction loss
        aux_loss: Auxiliary load balancing loss
    """
    # Get the standard next token prediction loss
    ntp_loss = outputs.loss if hasattr(outputs, 'loss') else None

    # Compute auxiliary loss from all MoE layers
    aux_loss = compute_moe_auxiliary_loss(model, router_aux_loss_coef)

    # Total loss
    if ntp_loss is not None:
        total_loss = ntp_loss + router_aux_loss_coef * aux_loss
    else:
        total_loss = router_aux_loss_coef * aux_loss

    return total_loss, ntp_loss, aux_loss

print("Auxiliary loss computation function defined")


print(" FIXED replace_ffn_with_moe function defined")
print("  - No expert noise - all experts start identical")
print("  - Proper weight extraction and copying")
print("  - Router initialized for top-2 routing")

In [ ]:
NUM_EXPERTS = 8  # Full 8 experts like Mixtral
NUM_EXPERTS_PER_TOK = 2  # Top-2 routing like Mixtral
ROUTER_JITTER_NOISE = 0.0
ROUTER_AUX_LOSS_COEF = 0.001  # FIXED: Reduced from 0.01 to allow expert 0 preference initially
USE_HALF_WIDTH = False  # FIXED: Use full intermediate_size (quantization makes this safe)


print("MoE Configuration ")

print(f"  Number of experts: {NUM_EXPERTS}")
print(f"  Experts per token: {NUM_EXPERTS_PER_TOK}")
print(f"  Half-width experts: {USE_HALF_WIDTH} (intermediate_size // 2)")
print(f"  Router jitter noise: {ROUTER_JITTER_NOISE}")
print(f"  Router aux loss coefficient: {ROUTER_AUX_LOSS_COEF}")

# Estimate memory with quantization
intermediate = 14336 // 2 if USE_HALF_WIDTH else 14336  # Full width
expert_params = NUM_EXPERTS * 32 * 3 * 4096 * intermediate  # experts × layers × projections × dims
expert_memory_bf16 = expert_params * 2 / 1e9  # bf16 = 2 bytes
expert_memory_8bit = expert_params * 1 / 1e9  # 8-bit = 1 byte
expert_memory_4bit = expert_params * 0.5 / 1e9  # 4-bit = 0.5 bytes
print(f"\nMemory Estimate:")
print(f"  Intermediate size: {intermediate} {'(half-width)' if USE_HALF_WIDTH else '(full)'}")
print(f"  Expert params: {expert_params/1e9:.2f}B")
print(f"  Expert memory (bf16): ~{expert_memory_bf16:.1f} GB  ← Without quantization (too large!)")
print(f"  Expert memory (8-bit): ~{expert_memory_8bit:.1f} GB  ← With 8-bit quantization")
print(f"  Expert memory (4-bit): ~{expert_memory_4bit:.1f} GB  ← With 4-bit quantization")

# ============================================================================
    #MEMORY CLEANUP BEFORE MOE CREATION
# ============================================================================
import gc
import torch

print("MEMORY CLEANUP BEFORE MOE CREATION")


# Check initial memory
gpu_initial = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before cleanup: {gpu_initial:.2f} GB")

# Delete base_model if it exists and is not the teacher
if 'base_model' in globals() and 'teacher_model' in globals():
    if base_model is not teacher_model:
        print("  Deleting base_model (keeping teacher_model for KD)")
        del base_model
    else:
        print("  base_model IS teacher_model, keeping it")
elif 'base_model' in globals():
    print("  Deleting base_model")
    del base_model

# Delete any other large model variables
vars_to_delete = ['model']
deleted_count = 0
for var_name in list(globals().keys()):
    if var_name in vars_to_delete and var_name in globals():
        obj = globals().get(var_name)
        if obj is not None and hasattr(obj, 'parameters'):
            try:
                next(obj.parameters())
                print(f"  Deleting: {var_name}")
                del globals()[var_name]
                deleted_count += 1
            except:
                pass

# Aggressive cleanup
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.synchronize()

# Try to trigger malloc_trim
try:
    import ctypes
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

gpu_after_cleanup = torch.cuda.memory_allocated() / 1e9
print(f"\nGPU memory after cleanup: {gpu_after_cleanup:.2f} GB")
print(f"Freed: {gpu_initial - gpu_after_cleanup:.2f} GB")

# CREATE 8-BIT QUANTIZATION CONFIG FOR EXPERTS

from transformers import BitsAndBytesConfig

# FIXED: Enable 8-bit quantization for experts
moe_expert_bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_quant_type="nf4",
)


from transformers import AutoModelForCausalLM

print("Loading fresh Mistral-7B model in bf16 (NO quantization for correct weight extraction)...")
moe_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # NO quantization_config - load in bf16 for correct weight extraction
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    )

gpu_with_base = torch.cuda.memory_allocated() / 1e9
print(f" Fresh model loaded")
print(f"  GPU memory: {gpu_with_base:.2f} GB")


print(f"Creating MoE model...")
print(f"  - {NUM_EXPERTS} experts, top-{NUM_EXPERTS_PER_TOK} routing")
print(f"  - Expert precision: bf16 (for correct weight copying)")
print(f"  - Intermediate size: {intermediate}")
print(f"  - CPU offload: Disabled (breaks gradients)")

# ============================================================================
# CRITICAL FIX: Use bnb_config=None for proper weight copying
# ============================================================================
# The quantized Linear4bit/Linear8bit layers don't support simple .weight.copy_()
# which breaks the expert weight initialization. Using regular nn.Linear (bf16)
# ensures weights are copied correctly from the original FFN.
#
# This preserves the pretrained model's knowledge in the MoE experts.
# ============================================================================

moe_model = replace_ffn_with_moe(
    model=moe_model,
    num_experts=NUM_EXPERTS,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK,
    router_jitter_noise=ROUTER_JITTER_NOISE,
    router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
    bnb_config=None,  # FIXED: Use bf16 experts for correct weight copying
    ram_threshold=80.0,                
    use_disk_offload=True,
    half_width=USE_HALF_WIDTH,
    enable_cpu_offload=False,           # Disabled (breaks training gradients)
)

# Final cleanup
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

gpu_final = torch.cuda.memory_allocated() / 1e9
gpu_reserved = torch.cuda.memory_reserved() / 1e9
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9


print(" MOE MODEL CREATED SUCCESSFULLY")

print(f"  GPU Memory:")
print(f"    Allocated: {gpu_final:.2f} GB")
print(f"    Reserved:  {gpu_reserved:.2f} GB")
print(f"    Total:     {gpu_total:.2f} GB")
print(f"    Usage:     {100*gpu_final/gpu_total:.1f}%")


print(f"    - Experts: {NUM_EXPERTS} ({NUM_EXPERTS_PER_TOK} active per token)")
print(f"    - Expert precision: bf16 (for correct weight copying)")
print(f"    - Attention layers: 8-bit quantized")
print(f"    - Router: bf16 (trainable)")
print(f"    - Load balancing coefficient: {ROUTER_AUX_LOSS_COEF}")
print(f"    - Router initialized to strongly prefer experts 0 & 1")

# Pretraining evaluation for dense MoE

In [ ]:
print("MOE BASELINE EVALUATION")


# 1. MMLU Accuracy
print("\n Evaluating MMLU accuracy...")
moe_results = evaluate_mmlu_comprehensive(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\n Computing FLOPs...")
moe_flops = compute_model_flops(moe_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print(" Measuring throughput...")
moe_throughput = compute_throughput_metrics(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print(" Analyzing parameter efficiency...")
moe_params = compute_parameter_efficiency(
    model=moe_model,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK
)

# 5. Memory metrics
print(" Collecting memory metrics...")
moe_memory = compute_memory_metrics(moe_model)

# Combine all metrics
moe_comprehensive = {
    **moe_results,
    'flops': moe_flops,
    **moe_throughput,
    **moe_params,
    **moe_memory,
}


print("MoE Baseline Evaluation Complete (Pre-LoRA)")
print(f"  Accuracy: {moe_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {moe_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {moe_comprehensive['ece']:.4f}")
print("\n Note: This is the TRUE baseline - no LoRA adapters applied yet")
print("   Next step: Apply QLoRA for training (Cell 15.5.2)")

# Training setup for baseline and training 

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset as TorchDataset
import torch

class MoETrainer(Trainer):
    """
    Custom Trainer for MoE models that computes auxiliary load balancing loss.
    Works with both regular models and PEFT-wrapped models.
    Total loss formula: L_total = L_NTP + λ * L_aux
    """

    def __init__(self, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.router_aux_loss_coef = router_aux_loss_coef

    def _moe_layers(self, model):
        """Get MoE layers from model, handling PEFT wrapper."""
        # Try different model structures (PEFT wrapped, base model, etc.)
        candidates = [
            lambda m: m.model.layers if hasattr(m, 'model') and hasattr(m.model, 'layers') else None,
            lambda m: m.base_model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'layers') else None,
            lambda m: m.base_model.model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'model') else None,
        ]
        
        for get_layers in candidates:
            try:
                layers = get_layers(model)
                if layers is not None:
                    return layers
            except:
                continue
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Enable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        # Forward pass
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        
        # Compute auxiliary load balancing loss
        aux_loss = self._compute_moe_auxiliary_loss(model)
        total_loss = ntp_loss + self.router_aux_loss_coef * aux_loss

        # Log losses periodically
        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "train/ntp_loss": ntp_loss.item(),
                "train/aux_loss": aux_loss.item(),
                "train/total_loss": total_loss.item(),
                "train/aux_loss_weight": self.router_aux_loss_coef,
            })

        # Disable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

    def _compute_moe_auxiliary_loss(self, model):
        """Compute auxiliary loss from all MoE layers."""
        device = next(model.parameters()).device
        aux_loss = torch.tensor(0.0, device=device)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()
        return aux_loss

# Training configuration for QLoRA
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,  # QLoRA standard learning rate
    "batch_size": 4,  # Larger batch with QLoRA
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "output_dir": "./mistral_moe_qlora",
}

print(" MoETrainer class defined (QLoRA compatible)\n")

In [ ]:
print("MoETrainer class defined\n")

# ============================================================================
# 15.5.2B - INTEGRATED MOE KD TRAINER
# ============================================================================

import torch.nn.functional as F

class IntegratedMoEKDTrainer(Trainer):
    """
    KD for MoE: L_total = (1-α)*L_NTP + α*L_KD + λ*L_aux + β*L_routingKD
    """
    def __init__(self, teacher_model=None, kd_config=None, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.router_aux_loss_coef = router_aux_loss_coef
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        self.routing_kd_weight = self.kd_config.get('routing_kd_weight', 0.0)
        self.enable_routing_kd = self.kd_config.get('enable_routing_kd', False)

    def _moe_layers(self, model):
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            return model.model.layers
        if hasattr(model, 'base_model') and hasattr(model.base_model.model, 'layers'):
            return model.base_model.model.layers
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss with KD.
        NOTE: Trainer automatically handles device placement - inputs are already on correct device!
        """
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        kd_loss = torch.tensor(0.0, device=device)
        routing_kd_loss = torch.tensor(0.0, device=device)

        if self.teacher_model is not None and self.kd_alpha > 0:
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

            if self.enable_routing_kd and self.routing_kd_weight > 0:
                ent = torch.tensor(0.0, device=device)
                layers_count = 0
                for layer in self._moe_layers(model):
                    mlp = getattr(layer, 'mlp', None)
                    if mlp is not None and hasattr(mlp, '_last_router_probs') and mlp._last_router_probs is not None:
                        probs = mlp._last_router_probs
                        ent_layer = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()
                        ent = ent + ent_layer
                        layers_count += 1
                if layers_count > 0:
                    routing_kd_loss = ent / layers_count

        aux_loss = torch.tensor(0.0, device=device)
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()

        total_loss = ((1 - self.kd_alpha) * ntp_loss
                      + self.kd_alpha * kd_loss
                      + self.router_aux_loss_coef * aux_loss
                      + self.routing_kd_weight * routing_kd_loss)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

print(" IntegratedMoEKDTrainer class defined")

In [ ]:
# ============================================================================
# TRAINING CELL - MoE Model Training with Optional Knowledge Distillation
# ============================================================================

import torch
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os

# --------------------------------------------------------------------------
# 1. CONFIGURATION - Choose your training mode
# --------------------------------------------------------------------------

# Training mode selection
USE_KNOWLEDGE_DISTILLATION = True  # Set to True for KD, False for standard training

# Dataset subset configuration
USE_SUBSET = True           # Use only a subset of the data
SUBSET_PERCENTAGE = 0.2     # Use 20% of the dataset

# MoE and training hyperparameters
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": 0.01,  # FIXED: Increased from 0.001
    "learning_rate": 2e-4,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.1,  # FIXED: Increased from 0.03
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 50,
    "save_steps": 100,
    "max_steps": 250,  # Train for 250 steps
    "max_length": 512,
}

# LoRA configuration - INCLUDES ROUTERS
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

# Knowledge distillation configuration
KD_CONFIG = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'enable_routing_kd': False,
    'router_aux_loss_coef': TRAINING_CONFIG['router_aux_loss_coef'],
    'name': 'Standard KD',
}

# Output directory
OUTPUT_DIR = "./mistral_moe_qlora_kd" if USE_KNOWLEDGE_DISTILLATION else "./mistral_moe_qlora"


In [ ]:
# --------------------------------------------------------------------------
# 3. SETUP MODELS
# --------------------------------------------------------------------------

student_model = moe_model

# Since the model was loaded in bf16 (NOT quantized with 4-bit/8-bit),
# we skip prepare_model_for_kbit_training - that function converts bf16->fp32
# which would double memory usage and cause OOM.
# Instead, just enable gradient checkpointing to save memory during training.

student_model.gradient_checkpointing_enable()
student_model.enable_input_require_grads()

print("Student model prepared for training")


# Dense Model Training with Knowledge Distillation

In [ ]:
# --------------------------------------------------------------------------
# 3. SETUP DATASETS
# --------------------------------------------------------------------------

# Check prerequisites and create tokenized datasets if needed
if 'tokenized_train' not in globals() or 'tokenized_eval' not in globals():
    # Check if we have the raw dataset
    if 'dataset' not in globals():
        raise ValueError(
            "❌ Missing prerequisites!\n"
            "Please run these cells in order:\n"
            "1. Dataset loading cell (creates 'dataset')\n"
            "2. Dataset tokenization cell (creates 'tokenized_train' and 'tokenized_eval')\n"
            "3. Then run this Dense KD training cell"
        )
    
    # We have dataset but not tokenized versions - create them now
    print("⚠️  tokenized_train/tokenized_eval not found. Creating them from 'dataset'...")
    print("   (This should normally be done in the tokenization cell)")
    
    if 'tokenizer' not in globals():
        raise ValueError("tokenizer must be defined! Run model loading cells first.")
    
    if 'TRAINING_CONFIG' not in globals():
        raise ValueError("TRAINING_CONFIG must be defined! Run configuration cells first.")
    
    def tokenize_function_for_training(examples):
        """Tokenize WITH answers for training."""
        return tokenizer(
            examples['formatted_prompt_with_answer'],
            truncation=True,
            max_length=TRAINING_CONFIG['max_length'],
            padding=False,
        )
    
    def tokenize_function_for_eval(examples):
        """Tokenize WITHOUT answers for evaluation."""
        return tokenizer(
            examples['formatted_prompt'],
            truncation=True,
            max_length=TRAINING_CONFIG['max_length'],
            padding=False,
        )
    
    print("   Tokenizing training data (with answers)...")
    tokenized_train = dataset['train'].map(
        tokenize_function_for_training,
        batched=True,
        remove_columns=dataset['train'].column_names,
        desc="Tokenizing training data"
    )
    
    print("   Tokenizing eval data (without answers)...")
    tokenized_eval = dataset['test'].map(
        tokenize_function_for_eval,
        batched=True,
        remove_columns=dataset['test'].column_names,
        desc="Tokenizing eval data"
    )
    
    print(f"✅ Created tokenized datasets: {len(tokenized_train):,} train, {len(tokenized_eval):,} eval")

# Use tokenized_train and tokenized_eval directly, or use train_dataset/val_dataset if they exist
if 'train_dataset' in globals() and train_dataset is not None:
    base_train_dataset = train_dataset
    print("Using existing train_dataset")
else:
    base_train_dataset = tokenized_train
    print("Using tokenized_train as base dataset")

if 'val_dataset' in globals() and val_dataset is not None:
    base_val_dataset = val_dataset
    print("Using existing val_dataset")
else:
    base_val_dataset = tokenized_eval
    print("Using tokenized_eval as base dataset")

# Apply subset if needed
if USE_SUBSET_DENSE:
    subset_size = int(len(base_train_dataset) * SUBSET_PERCENTAGE_DENSE)
    train_dataset_subset_dense = base_train_dataset.select(range(subset_size))
    print(f" Using {SUBSET_PERCENTAGE_DENSE*100}% of training data: {len(train_dataset_subset_dense)} samples")
else:
    train_dataset_subset_dense = base_train_dataset

# For evaluation, use val_dataset or tokenized_eval
if 'eval_dataset' in globals() and eval_dataset is not None:
    eval_dataset_dense = eval_dataset
elif 'val_dataset' in globals() and val_dataset is not None:
    eval_dataset_dense = val_dataset
else:
    eval_dataset_dense = base_val_dataset
    print("Using tokenized_eval for evaluation")

In [ ]:
# ============================================================================
# DENSE MODEL KNOWLEDGE DISTILLATION TRAINER
# ============================================================================

from transformers import Trainer
import torch.nn.functional as F
import torch

class DenseKDTrainer(Trainer):
    """
    Knowledge Distillation Trainer for Dense Models.
    Loss: L_total = (1-α)*L_NTP + α*L_KD
    
    Where:
    - L_NTP: Next Token Prediction loss (standard supervised learning)
    - L_KD: Knowledge Distillation loss (KL divergence between teacher and student logits)
    - α: Weighting factor for KD loss (typically 0.5)
    """
    def __init__(self, teacher_model=None, kd_config=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        
        if self.teacher_model is not None:
            self.teacher_model.eval()
            for param in self.teacher_model.parameters():
                param.requires_grad = False
            print(f" Teacher model loaded (frozen, KD alpha={self.kd_alpha}, T={self.temperature})")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss with Knowledge Distillation.
        
        Args:
            model: The student model
            inputs: Input dictionary with input_ids, attention_mask, labels, etc.
            return_outputs: Whether to return model outputs along with loss
            num_items_in_batch: Number of items in batch (for newer transformers versions, optional)
        """
        # Forward pass through student model
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        # Initialize KD loss
        kd_loss = torch.tensor(0.0, device=device)

        # Compute KD loss if teacher model is available
        if self.teacher_model is not None and self.kd_alpha > 0:
            # Move inputs to teacher's device
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            
            # Forward pass through teacher model (no gradients)
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            # Apply temperature scaling
            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            # KL divergence loss
            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

        # Combined loss
        total_loss = (1 - self.kd_alpha) * ntp_loss + self.kd_alpha * kd_loss

        return (total_loss, outputs) if return_outputs else total_loss

print(" DenseKDTrainer class defined")

In [ ]:
# ============================================================================
# TRAINING SETUP: DENSE MODEL WITH KNOWLEDGE DISTILLATION
# (NO CALLBACK - EVALUATION ONLY AT END)
# ============================================================================

from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os
import torch
import time

# --------------------------------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------------------------------

# Training mode
USE_KNOWLEDGE_DISTILLATION_DENSE = True  # Set to True for KD, False for standard training

# Dataset subset configuration
USE_SUBSET_DENSE = True
SUBSET_PERCENTAGE_DENSE = 0.2  # Use 20% of the dataset

# Training hyperparameters (only valid TrainingArguments parameters)
TRAINING_CONFIG_DENSE = {
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 4,  # Fixed: changed from "batch_size"
    "per_device_eval_batch_size": 4,    # Added for evaluation
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.1,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "save_steps": 100,
    "max_steps": 250,  # Train for 250 steps
    "fp16": True,
    "bf16": False,
    "save_total_limit": 2,
}

# Max length for tokenization (not for TrainingArguments)
MAX_LENGTH_DENSE = 512

# LoRA configuration
LORA_CONFIG_DENSE = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "task_type": TaskType.CAUSAL_LM,
}

# Knowledge distillation configuration
KD_CONFIG_DENSE = {
    'kd_alpha': 0.5,        # Weight for KD loss (0.5 = equal weight with NTP loss)
    'temperature': 4.0,     # Temperature for softmax (higher = softer distribution)
    'name': 'Standard KD',
}

# Output directory
OUTPUT_DIR_DENSE = "./dense_model_kd" if USE_KNOWLEDGE_DISTILLATION_DENSE else "./dense_model_standard"

print("=" * 80)
print("DENSE MODEL TRAINING CONFIGURATION")
print("=" * 80)
print(f"Training Mode: {'Knowledge Distillation' if USE_KNOWLEDGE_DISTILLATION_DENSE else 'Standard'}")
print(f"KD Alpha: {KD_CONFIG_DENSE['kd_alpha']}")
print(f"Temperature: {KD_CONFIG_DENSE['temperature']}")
print(f"Output Directory: {OUTPUT_DIR_DENSE}")
print(f"Max Steps: {TRAINING_CONFIG_DENSE['max_steps']}")
print("=" * 80)

# --------------------------------------------------------------------------
# 2. SETUP MODELS
# --------------------------------------------------------------------------

# Teacher model: Use the baseline dense model (already evaluated)
if 'teacher_model' not in globals() or teacher_model is None:
    # Fallback: use base_model as teacher
    if 'base_model' in globals():
        teacher_model = base_model.eval()
        for param in teacher_model.parameters():
            param.requires_grad = False
        print("\n Teacher model set from base_model")
    else:
        raise ValueError("teacher_model or base_model must be available!")

# Student model: Load fresh model for training
print("\n Setting up student model...")

# Load base model with quantization (same as baseline)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Quantization config (same as your baseline setup)
bnb_config_dense = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Get model name from your existing setup
if 'base_model' in globals() and hasattr(base_model, 'config'):
    MODEL_NAME_DENSE = base_model.config._name_or_path if hasattr(base_model.config, '_name_or_path') else "mistralai/Mistral-7B-v0.1"
else:
    MODEL_NAME_DENSE = "mistralai/Mistral-7B-v0.1"

print(f"Loading student model from: {MODEL_NAME_DENSE}")

student_model_dense = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_DENSE,
    quantization_config=bnb_config_dense,
    device_map="auto",
    trust_remote_code=True,
)

student_model_dense.config.use_cache = False

# Prepare model for k-bit training
student_model_dense = prepare_model_for_kbit_training(student_model_dense)
print(" Student model prepared for k-bit training")

# Apply LoRA
peft_config_dense = LoraConfig(**LORA_CONFIG_DENSE)
student_model_dense = get_peft_model(student_model_dense, peft_config_dense)
print(" LoRA adapters applied")

# Enable gradient checkpointing (set on model, not in TrainingArguments)
student_model_dense.gradient_checkpointing_enable()
student_model_dense.train()
print(" Student model set to training mode")

# Print trainable parameters
trainable_params = sum(p.numel() for p in student_model_dense.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in student_model_dense.parameters())
print(f" Trainable parameters: {trainable_params/1e6:.2f}M / {total_params/1e9:.2f}B ({100*trainable_params/total_params:.2f}%)")

# --------------------------------------------------------------------------
# 3. SETUP DATASETS
# --------------------------------------------------------------------------

# Check prerequisites and create tokenized datasets if needed
if 'tokenized_train' not in globals() or 'tokenized_eval' not in globals():
    # Check if we have the raw dataset
    if 'dataset' not in globals():
        raise ValueError(
            "❌ Missing prerequisites!\n"
            "Please run these cells in order:\n"
            "1. Dataset loading cell (creates 'dataset')\n"
            "2. Dataset tokenization cell (creates 'tokenized_train' and 'tokenized_eval')\n"
            "3. Then run this Dense KD training cell"
        )
    
    # We have dataset but not tokenized versions - create them now
    print("⚠️  tokenized_train/tokenized_eval not found. Creating them from 'dataset'...")
    print("   (This should normally be done in the tokenization cell)")
    
    if 'tokenizer' not in globals():
        raise ValueError("tokenizer must be defined! Run model loading cells first.")
    
    # Get max_length from MAX_LENGTH_DENSE (defined in this cell) or TRAINING_CONFIG or use default
    if 'MAX_LENGTH_DENSE' in globals():
        max_length = MAX_LENGTH_DENSE
    elif 'TRAINING_CONFIG_DENSE' in globals() and 'max_length' in TRAINING_CONFIG_DENSE:
        max_length = TRAINING_CONFIG_DENSE['max_length']
    elif 'TRAINING_CONFIG' in globals() and 'max_length' in TRAINING_CONFIG:
        max_length = TRAINING_CONFIG['max_length']
    else:
        max_length = 512
        print("   Warning: Using default max_length=512")
    
    def tokenize_function_for_training(examples):
        """Tokenize WITH answers for training."""
        return tokenizer(
            examples['formatted_prompt_with_answer'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    
    def tokenize_function_for_eval(examples):
        """Tokenize WITHOUT answers for evaluation."""
        return tokenizer(
            examples['formatted_prompt'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    
    print("   Tokenizing training data (with answers)...")
    tokenized_train = dataset['train'].map(
        tokenize_function_for_training,
        batched=True,
        remove_columns=dataset['train'].column_names,
        desc="Tokenizing training data"
    )
    
    print("   Tokenizing eval data (without answers)...")
    tokenized_eval = dataset['test'].map(
        tokenize_function_for_eval,
        batched=True,
        remove_columns=dataset['test'].column_names,
        desc="Tokenizing eval data"
    )
    
    print(f"✅ Created tokenized datasets: {len(tokenized_train):,} train, {len(tokenized_eval):,} eval")

# Use tokenized_train and tokenized_eval directly, or use train_dataset/val_dataset if they exist
if 'train_dataset' in globals() and train_dataset is not None:
    base_train_dataset = train_dataset
    print("Using existing train_dataset")
else:
    base_train_dataset = tokenized_train
    print("Using tokenized_train as base dataset")

if 'val_dataset' in globals() and val_dataset is not None:
    base_val_dataset = val_dataset
    print("Using existing val_dataset")
else:
    base_val_dataset = tokenized_eval
    print("Using tokenized_eval as base dataset")

# Apply subset if needed
if USE_SUBSET_DENSE:
    subset_size = int(len(base_train_dataset) * SUBSET_PERCENTAGE_DENSE)
    train_dataset_subset_dense = base_train_dataset.select(range(subset_size))
    print(f" Using {SUBSET_PERCENTAGE_DENSE*100}% of training data: {len(train_dataset_subset_dense)} samples")
else:
    train_dataset_subset_dense = base_train_dataset

# For evaluation, use val_dataset or tokenized_eval
if 'eval_dataset' in globals() and eval_dataset is not None:
    eval_dataset_dense = eval_dataset
elif 'val_dataset' in globals() and val_dataset is not None:
    eval_dataset_dense = val_dataset
else:
    eval_dataset_dense = base_val_dataset
    print("Using tokenized_eval for evaluation")

# --------------------------------------------------------------------------
# 4. SETUP TRAINING ARGUMENTS
# --------------------------------------------------------------------------

training_args_dense = TrainingArguments(
    output_dir=OUTPUT_DIR_DENSE,
    **TRAINING_CONFIG_DENSE,
    save_strategy="steps",
    load_best_model_at_end=False,
    report_to="wandb" if 'wandb' in globals() else None,
    run_name=f"dense_kd_{KD_CONFIG_DENSE['kd_alpha']}" if USE_KNOWLEDGE_DISTILLATION_DENSE else "dense_standard",
    logging_first_step=True,
    logging_strategy="steps",
    eval_strategy="no",  # No evaluation during training
)

# --------------------------------------------------------------------------
# 5. SETUP DATA COLLATOR
# --------------------------------------------------------------------------

data_collator_dense = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --------------------------------------------------------------------------
# 6. SETUP TRAINER (NO CALLBACKS)
# --------------------------------------------------------------------------

print("\n🚀 Initializing trainer...")

if USE_KNOWLEDGE_DISTILLATION_DENSE:
    print("  Using DenseKDTrainer for knowledge distillation")
    print("  No evaluation callback - evaluation will be done post-training only")
    trainer_dense = DenseKDTrainer(
        model=student_model_dense,
        args=training_args_dense,
        train_dataset=train_dataset_subset_dense,
        eval_dataset=eval_dataset_dense,
        data_collator=data_collator_dense,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG_DENSE,
        # NO CALLBACKS - evaluation only at end
    )
else:
    print("  Using standard Trainer")
    print("  No evaluation callback - evaluation will be done post-training only")
    from transformers import Trainer
    trainer_dense = Trainer(
        model=student_model_dense,
        args=training_args_dense,
        train_dataset=train_dataset_subset_dense,
        eval_dataset=eval_dataset_dense,
        data_collator=data_collator_dense,
        # NO CALLBACKS - evaluation only at end
    )

print(" Trainer initialized successfully!")

In [ ]:
# --------------------------------------------------------------------------
# 7. START TRAINING
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING DENSE MODEL TRAINING")
print("=" * 80)
print("Note: No evaluation during training - will evaluate after training completes")

# Clear GPU cache
torch.cuda.empty_cache()
import gc
gc.collect()

# Start training
train_result_dense = trainer_dense.train()

print("\n" + "=" * 80)
print("DENSE MODEL TRAINING COMPLETE")
print("=" * 80)

# Print final training metrics
print("\nFinal training metrics:")
for key, value in train_result_dense.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save the final model
print(f"\n💾 Saving model to {OUTPUT_DIR_DENSE}/final_model...")
trainer_dense.save_model(f"{OUTPUT_DIR_DENSE}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR_DENSE}/final_model")
print("✅ Model saved successfully!")

# Store student model for evaluation
dense_kd_model = student_model_dense

# Post-Training Evaluation: Knowledge-Distilled Dense Model

In [ ]:
# ============================================================================
# POST-TRAINING EVALUATION: KNOWLEDGE-DISTILLED DENSE MODEL
# ============================================================================

import json
import os
import numpy as np

print("\n" + "=" * 80)
print("POST-TRAINING EVALUATION: KNOWLEDGE-DISTILLED DENSE MODEL")
print("=" * 80)

# Load trained model (or use in-memory model)
if 'dense_kd_model' in globals():
    eval_model_dense = dense_kd_model
    eval_model_dense.eval()
    print(" Using in-memory trained model")
else:
    # Load from checkpoint
    from peft import PeftModel
    eval_model_dense = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_DENSE,
        quantization_config=bnb_config_dense,
        device_map="auto",
        trust_remote_code=True,
    )
    eval_model_dense = PeftModel.from_pretrained(eval_model_dense, f"{OUTPUT_DIR_DENSE}/final_model")
    eval_model_dense.eval()
    print(f" Loaded model from {OUTPUT_DIR_DENSE}/final_model")

# Comprehensive evaluation (same metrics as baseline)
print("\n Running comprehensive evaluation...")

# 1. MMLU Accuracy
print("  1. Computing MMLU Accuracy...")
kd_dense_results = evaluate_mmlu_comprehensive(
    model=eval_model_dense,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs
print("  2. Computing FLOPs...")
kd_dense_flops = compute_model_flops(eval_model_dense, seq_length=MAX_LENGTH)

# 3. Throughput
print("  3. Measuring throughput...")
kd_dense_throughput = compute_throughput_metrics(
    model=eval_model_dense,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=1000
)

# 4. Parameter efficiency
print("  4. Analyzing parameter efficiency...")
kd_dense_params = compute_parameter_efficiency(
    model=eval_model_dense,
    num_experts_per_tok=1  # Dense model
)

# 5. Memory metrics
print("  5. Collecting memory metrics...")
kd_dense_memory = compute_memory_metrics(eval_model_dense)

# Combine all metrics
kd_dense_comprehensive = {
    **kd_dense_results,
    'flops': kd_dense_flops,
    **kd_dense_throughput,
    **kd_dense_params,
    **kd_dense_memory,
}

# Display comprehensive results
print("\n" + "=" * 80)
print("COMPREHENSIVE KNOWLEDGE-DISTILLED DENSE MODEL METRICS")
print("=" * 80)

print("\n Accuracy Metrics:")
print(f"  MMLU Accuracy: {kd_dense_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {kd_dense_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {kd_dense_comprehensive['ece']:.4f}")

print("\n Computational Efficiency:")
print(f"  FLOPs per forward pass: {kd_dense_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {kd_dense_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {kd_dense_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {kd_dense_comprehensive['samples_per_second']:.2f}")

print("\n Parameter Efficiency:")
print(f"  Total parameters: {kd_dense_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {kd_dense_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {kd_dense_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {kd_dense_comprehensive['sparsity_ratio']:.2%}")

print("\n Memory Usage:")
print(f"  Model size: {kd_dense_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {kd_dense_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {kd_dense_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# Comparison with baseline
if 'baseline_comprehensive' in globals():
    print("\n" + "=" * 80)
    print("COMPARISON WITH BASELINE DENSE MODEL")
    print("=" * 80)
    
    acc_diff = kd_dense_comprehensive['accuracy'] - baseline_comprehensive['accuracy']
    acc_change = (acc_diff / baseline_comprehensive['accuracy']) * 100
    
    print("\n Accuracy Comparison:")
    print(f"  Baseline Accuracy:     {baseline_comprehensive['accuracy']:.4f}")
    print(f"  KD Dense Accuracy:     {kd_dense_comprehensive['accuracy']:.4f}")
    print(f"  Difference:            {acc_diff:+.4f} ({acc_change:+.2f}%)")
    
    print("\n⚡ Efficiency Comparison:")
    print(f"  Baseline Tokens/sec:   {baseline_comprehensive['tokens_per_second']:.2f}")
    print(f"  KD Dense Tokens/sec:    {kd_dense_comprehensive['tokens_per_second']:.2f}")
    diff_tokens = kd_dense_comprehensive['tokens_per_second'] - baseline_comprehensive['tokens_per_second']
    print(f"  Difference:            {diff_tokens:+.2f} ({100*diff_tokens/baseline_comprehensive['tokens_per_second']:+.2f}%)")
    
    print("\n Memory Comparison:")
    print(f"  Baseline GPU Memory:   {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  KD Dense GPU Memory:   {kd_dense_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    diff_mem = kd_dense_comprehensive['gpu_memory_allocated_gb'] - baseline_comprehensive['gpu_memory_allocated_gb']
    print(f"  Difference:            {diff_mem:+.2f} GB")

# Save results
os.makedirs("results", exist_ok=True)
with open("results/dense_kd_comprehensive.json", 'w') as f:
    json.dump({k: v for k, v in kd_dense_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)

# Log to wandb if available
try:
    if wandb.run is not None:
        wandb.log({
            'dense_kd/accuracy': kd_dense_comprehensive['accuracy'],
            'dense_kd/top2_accuracy': kd_dense_comprehensive['top2_accuracy'],
            'dense_kd/ece': kd_dense_comprehensive['ece'],
            'dense_kd/flops_billions': kd_dense_comprehensive['flops']/1e9,
            'dense_kd/tokens_per_second': kd_dense_comprehensive['tokens_per_second'],
            'dense_kd/ms_per_token': kd_dense_comprehensive['ms_per_token'],
            'dense_kd/active_params_billions': kd_dense_comprehensive['active_params']/1e9,
            'dense_kd/gpu_memory_gb': kd_dense_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

print("\n Post-training evaluation complete!")
print(" Results saved to results/dense_kd_comprehensive.json")

In [ ]:
# --------------------------------------------------------------------------
# SETUP TRAINING ARGUMENTS
# --------------------------------------------------------------------------

from transformers import TrainingArguments
import os

print("\nSetting up training arguments...")

# Use OUTPUT_DIR from configuration (set to "./mistral_moe_qlora_kd" for KD)
output_dir = OUTPUT_DIR if 'OUTPUT_DIR' in globals() else "./mistral_moe_qlora"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    max_steps=TRAINING_CONFIG['max_steps'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    logging_dir=f"{output_dir}/logs",
    logging_steps=TRAINING_CONFIG['logging_steps'],
    eval_strategy="no",  # Disable evaluation during training
    save_strategy="steps",
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=False,  # No evaluation, so no best model to load
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to=["tensorboard"],
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

print("Training arguments configured:")
print(f"  Output directory: {output_dir}")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Warmup ratio: {TRAINING_CONFIG['warmup_ratio']}")
print(f"  Evaluation during training: DISABLED")

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("\nInitializing trainer...")

# Check which training dataset variable exists
if 'train_dataset' in globals() and train_dataset is not None:
    train_dataset_to_use = train_dataset
    print("  Using train_dataset for training")
elif 'tokenized_train' in globals() and tokenized_train is not None:
    train_dataset_to_use = tokenized_train
    print("  Using tokenized_train for training")
else:
    raise ValueError("Neither 'train_dataset' nor 'tokenized_train' found. Please run dataset tokenization cell first.")

# Select trainer based on configuration
if USE_KNOWLEDGE_DISTILLATION:
    print("  Using IntegratedMoEKDTrainer for knowledge distillation")
    trainer = IntegratedMoEKDTrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset_to_use,
        eval_dataset=None,  # No evaluation during training
        data_collator=data_collator,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        # No callbacks - evaluation will be done after training
    )
else:
    print("  Using MoETrainer for standard training")
    trainer = MoETrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset_to_use,
        eval_dataset=None,  # No evaluation during training
        data_collator=data_collator,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        # No callbacks - evaluation will be done after training
    )

print("Trainer initialized successfully!")
print("  Note: Evaluation is completely disabled - will only train")
print("  Run evaluation separately after training completes")

In [ ]:
# --------------------------------------------------------------------------
# TRAIN THE MODEL
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)

# Clear GPU cache before training
import gc
torch.cuda.empty_cache()
gc.collect()

print(f"\nTraining configuration:")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Eval steps: {TRAINING_CONFIG['eval_steps']}")
print(f"  Knowledge Distillation: {USE_KNOWLEDGE_DISTILLATION}")
if USE_KNOWLEDGE_DISTILLATION:
    print(f"  KD Alpha: {KD_CONFIG['kd_alpha']}")
    print(f"  Temperature: {KD_CONFIG['temperature']}")

# Start training
train_result = trainer.train()

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

# Print final training metrics
print("\nFinal training metrics:")
for key, value in train_result.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save the final model
print(f"\nSaving model to {OUTPUT_DIR}/final_model...")
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print("Model saved successfully!")

In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION: MoE MODEL WITH KNOWLEDGE DISTILLATION
# ============================================================================

import json
import os
import torch

print("=" * 80)
print("COMPREHENSIVE EVALUATION: MoE MODEL WITH KNOWLEDGE DISTILLATION")
print("=" * 80)

# --------------------------------------------------------------------------
# 1. Load Trained MoE KD Model
# --------------------------------------------------------------------------

print("\n📦 Loading MoE KD model...")

# Determine which model to evaluate
if 'student_model' in globals() and student_model is not None and USE_KNOWLEDGE_DISTILLATION:
    moe_kd_model = student_model
    model_name = "MoE KD Model (in-memory)"
    print(" ✓ Using student_model from training session (KD enabled)")
else:
    # Load from checkpoint if not in memory
    from peft import PeftModel
    
    # Try to load from the KD checkpoint directory
    checkpoint_dir = "./mistral_moe_qlora_kd" if 'OUTPUT_DIR' not in globals() else OUTPUT_DIR
    
    print(f"Loading from checkpoint: {checkpoint_dir}/final_model")
    
    # Load base MoE model first (assumes moe_model is still in memory)
    if 'moe_model' not in globals():
        raise ValueError("moe_model not found. Please load the base MoE model first.")
    
    moe_kd_model = moe_model
    
    # Load LoRA weights from KD training
    moe_kd_model = PeftModel.from_pretrained(moe_kd_model, f"{checkpoint_dir}/final_model")
    model_name = f"MoE KD Model from {checkpoint_dir}"
    print(f" ✓ Loaded checkpoint: {checkpoint_dir}/final_model")

# Set to eval mode
moe_kd_model.eval()
print(f" ✓ Model set to evaluation mode")

# --------------------------------------------------------------------------
# 2. MMLU Accuracy Evaluation
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. MMLU ACCURACY EVALUATION")
print("=" * 80)

print("Evaluating MMLU accuracy...")
moe_kd_results = evaluate_mmlu_comprehensive(
    model=moe_kd_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

print(f" ✓ MMLU Accuracy: {moe_kd_results['accuracy']:.4f}")
print(f" ✓ Top-2 Accuracy: {moe_kd_results['top2_accuracy']:.4f}")
print(f" ✓ ECE: {moe_kd_results['ece']:.4f}")

# --------------------------------------------------------------------------
# 3. FLOPs Estimation
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. COMPUTATIONAL EFFICIENCY (FLOPs)")
print("=" * 80)

print("Computing FLOPs...")
moe_kd_flops = compute_model_flops(moe_kd_model, seq_length=MAX_LENGTH)
print(f" ✓ FLOPs per forward pass: {moe_kd_flops/1e9:.2f}G")

# --------------------------------------------------------------------------
# 4. Throughput Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. THROUGHPUT METRICS")
print("=" * 80)

print("Measuring throughput...")
moe_kd_throughput = compute_throughput_metrics(
    model=moe_kd_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

print(f" ✓ Tokens/second: {moe_kd_throughput['tokens_per_second']:.2f}")
print(f" ✓ ms/token: {moe_kd_throughput['ms_per_token']:.3f}")
print(f" ✓ Samples/second: {moe_kd_throughput['samples_per_second']:.2f}")

# --------------------------------------------------------------------------
# 5. Parameter Efficiency
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. PARAMETER EFFICIENCY")
print("=" * 80)

print("Analyzing parameter efficiency...")

# Determine num_experts_per_tok from model
base_model_ref = moe_kd_model
if hasattr(moe_kd_model, 'base_model'):
    base_model_ref = moe_kd_model.base_model
if hasattr(base_model_ref, 'model'):
    base_model_ref = base_model_ref.model

# Get num_experts_per_tok from first MoE layer
num_experts_per_tok_value = NUM_EXPERTS_PER_TOK if 'NUM_EXPERTS_PER_TOK' in globals() else 2
if hasattr(base_model_ref, 'layers') and len(base_model_ref.layers) > 0:
    first_layer = base_model_ref.layers[0]
    if hasattr(first_layer, 'mlp') and hasattr(first_layer.mlp, 'num_experts_per_tok'):
        num_experts_per_tok_value = first_layer.mlp.num_experts_per_tok

moe_kd_params = compute_parameter_efficiency(
    model=moe_kd_model,
    num_experts_per_tok=num_experts_per_tok_value
)

print(f" ✓ Total parameters: {moe_kd_params['total_params']/1e9:.2f}B")
print(f" ✓ Active parameters: {moe_kd_params['active_params']/1e9:.2f}B")
print(f" ✓ Trainable parameters: {moe_kd_params['trainable_params']/1e6:.2f}M")
print(f" ✓ Sparsity ratio: {moe_kd_params['sparsity_ratio']:.2%}")

# --------------------------------------------------------------------------
# 6. Memory Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. MEMORY USAGE")
print("=" * 80)

print("Collecting memory metrics...")
moe_kd_memory = compute_memory_metrics(moe_kd_model)

print(f" ✓ Model size: {moe_kd_memory['model_size_mb']:.2f} MB")
print(f" ✓ GPU allocated: {moe_kd_memory['gpu_memory_allocated_gb']:.2f} GB")
if moe_kd_memory.get('gpu_memory_reserved_gb') is not None:
    print(f" ✓ GPU reserved: {moe_kd_memory['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 7. Combine All Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINING ALL METRICS")
print("=" * 80)

moe_kd_comprehensive = {
    **moe_kd_results,
    'flops': moe_kd_flops,
    **moe_kd_throughput,
    **moe_kd_params,
    **moe_kd_memory,
}

# --------------------------------------------------------------------------
# 8. Display Comprehensive Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPREHENSIVE MoE KNOWLEDGE DISTILLATION METRICS")
print("=" * 80)

print("\n📊 Accuracy Metrics:")
print(f"  MMLU Accuracy: {moe_kd_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {moe_kd_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {moe_kd_comprehensive['ece']:.4f}")

print("\n⚡ Computational Efficiency:")
print(f"  FLOPs per forward pass: {moe_kd_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {moe_kd_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {moe_kd_comprehensive['ms_per_token']:.3f}")
print(f"  Samples/second: {moe_kd_comprehensive['samples_per_second']:.2f}")

print("\n🔧 Parameter Efficiency:")
print(f"  Total parameters: {moe_kd_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {moe_kd_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {moe_kd_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {moe_kd_comprehensive['sparsity_ratio']:.2%}")

print("\n💾 Memory Usage:")
print(f"  Model size: {moe_kd_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {moe_kd_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
if moe_kd_comprehensive.get('gpu_memory_reserved_gb') is not None:
    print(f"  GPU reserved: {moe_kd_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 9. Save Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create results directory if it doesn't exist
os.makedirs("results", exist_ok=True)

# Save comprehensive results
results_file = "results/moe_kd_comprehensive.json"
with open(results_file, 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    def convert_to_serializable(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_serializable(item) for item in obj]
        return obj
    
    serializable_results = convert_to_serializable(moe_kd_comprehensive)
    json.dump(serializable_results, f, indent=2)

print(f" ✓ Saved comprehensive results to: {results_file}")

# --------------------------------------------------------------------------
# 10. Summary
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)
print(f"\n✓ All metrics computed and saved for MoE Knowledge Distillation model")
print(f"✓ Results file: {results_file}")
print(f"\nKey Metrics Summary:")
print(f"  Accuracy: {moe_kd_comprehensive['accuracy']:.4f}")
print(f"  Tokens/sec: {moe_kd_comprehensive['tokens_per_second']:.2f}")
print(f"  FLOPs: {moe_kd_comprehensive['flops']/1e9:.2f}G")
print(f"  Parameters: {moe_kd_comprehensive['total_params']/1e9:.2f}B")

# Model Comparison Table

In [ ]:
# ============================================================================
# COMPREHENSIVE MODEL COMPARISON TABLE
# ============================================================================

import json
import os
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
import torch

def compute_perplexity_from_model(model, tokenizer, eval_dataset, device="cuda", max_samples=100):
    """Compute perplexity by evaluating model on eval dataset."""
    model.eval()
    total_loss = 0.0
    num_samples = 0
    
    eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=False)
    
    with torch.no_grad():
        for i, batch in enumerate(eval_dataloader):
            if i >= max_samples:
                break
            
            # Process batch
            processed_batch = {}
            for k, v in batch.items():
                if isinstance(v, list):
                    v = torch.tensor(v)
                if v.dim() == 1:
                    v = v.unsqueeze(0)
                processed_batch[k] = v.to(device)
            
            if 'labels' not in processed_batch and 'input_ids' in processed_batch:
                processed_batch['labels'] = processed_batch['input_ids'].clone()
            
            outputs = model(**processed_batch)
            
            if outputs.loss is not None:
                total_loss += outputs.loss.item()
                num_samples += 1
    
    avg_loss = total_loss / num_samples if num_samples > 0 else 0
    perplexity = np.exp(avg_loss) if avg_loss > 0 else 0.0
    
    return perplexity, avg_loss

def load_or_compute_metrics(model_name, results_file=None, model=None, tokenizer=None, eval_dataset=None):
    """Load metrics from file or compute if not available."""
    metrics = {}
    
    # Try to load from file
    if results_file and os.path.exists(results_file):
        try:
            with open(results_file, 'r') as f:
                metrics = json.load(f)
            print(f"Loaded {model_name} metrics from {results_file}")
        except Exception as e:
            print(f"Could not load {model_name} from {results_file}: {e}")
    
    # Compute perplexity if missing
    if 'perplexity' not in metrics or metrics.get('perplexity') == 0:
        if model is not None and tokenizer is not None and eval_dataset is not None:
            print(f"  Computing perplexity for {model_name}...")
            try:
                perplexity, eval_loss = compute_perplexity_from_model(model, tokenizer, eval_dataset)
                metrics['perplexity'] = perplexity
                metrics['eval_loss'] = eval_loss
                print(f"  Perplexity computed: {perplexity:.2f}")
            except Exception as e:
                print(f"  Could not compute perplexity: {e}")
                metrics['perplexity'] = None
        else:
            metrics['perplexity'] = None
    
    return metrics

# Load all model results
print("=" * 80)
print("LOADING MODEL RESULTS FOR COMPARISON")
print("=" * 80)

# 1. Dense Baseline
dense_baseline = load_or_compute_metrics(
    "Dense Baseline",
    results_file="results/baseline_comprehensive.json",
    model=base_model if 'base_model' in globals() else None,
    tokenizer=tokenizer if 'tokenizer' in globals() else None,
    eval_dataset=val_dataset if 'val_dataset' in globals() else None
)

# 2. Dense with KD
dense_kd = load_or_compute_metrics(
    "Dense with KD",
    results_file="results/dense_kd_comprehensive.json",
    model=dense_kd_model if 'dense_kd_model' in globals() else None,
    tokenizer=tokenizer if 'tokenizer' in globals() else None,
    eval_dataset=val_dataset if 'val_dataset' in globals() else None
)

# 3. MoE with Standard Training
# Check if we have a separate MoE standard training results file
moe_standard_file = "results/moe_standard_comprehensive.json"
if not os.path.exists(moe_standard_file):
    # Try to use trained_moe_comprehensive.json - you may need to adjust this
    # based on which training run it represents
    moe_standard_file = None  # Will try to compute from model if available

moe_standard = load_or_compute_metrics(
    "MoE Standard Training",
    results_file=moe_standard_file,
    model=student_model if 'student_model' in globals() and not USE_KNOWLEDGE_DISTILLATION else None,
    tokenizer=tokenizer if 'tokenizer' in globals() else None,
    eval_dataset=val_dataset if 'val_dataset' in globals() else None
)

# 4. MoE with KD
moe_kd = load_or_compute_metrics(
    "MoE with KD",
    results_file="results/trained_moe_comprehensive.json",
    model=student_model if 'student_model' in globals() and USE_KNOWLEDGE_DISTILLATION else None,
    tokenizer=tokenizer if 'tokenizer' in globals() else None,
    eval_dataset=val_dataset if 'val_dataset' in globals() else None
)

print("\n" + "=" * 80)
print("RESULTS LOADED")
print("=" * 80)


In [ ]:
# ============================================================================
# UNIFIED MODEL COMPARISON TABLE
# ============================================================================

import json
import os
import pandas as pd
import numpy as np

def format_value(value, format_type='float', decimals=4):
    """Format value based on type."""
    if value is None:
        return "N/A"
    
    if format_type == 'float':
        return f"{value:.{decimals}f}"
    elif format_type == 'percent':
        return f"{value*100:.2f}%"
    elif format_type == 'scientific':
        return f"{value:.2e}"
    elif format_type == 'int':
        return f"{int(value):,}"
    elif format_type == 'billions':
        return f"{value/1e9:.2f}B"
    elif format_type == 'millions':
        return f"{value/1e6:.2f}M"
    elif format_type == 'thousands':
        return f"{value/1e3:.2f}K"
    else:
        return str(value)

# Try to load from files first, then use provided values as fallback
print("=" * 80)
print("LOADING MODEL RESULTS FOR UNIFIED COMPARISON")
print("=" * 80)

# 1. Dense Baseline - Load from file or use provided values
dense_baseline = {}
if os.path.exists("results/baseline_comprehensive.json"):
    try:
        with open("results/baseline_comprehensive.json", 'r') as f:
            dense_baseline = json.load(f)
        print("Loaded Dense Baseline from results/baseline_comprehensive.json")
    except:
        pass

# If file doesn't exist or is incomplete, use provided values
if not dense_baseline or 'accuracy' not in dense_baseline:
    print("Using provided Dense Baseline values")
    dense_baseline = {
        'accuracy': 0.6590,
        'top2_accuracy': 0.8370,
        'ece': 0.0796,
        'perplexity': None,
        'flops': 8796093022208,  # 8796.09G
        'tokens_per_second': 3071.60,
        'ms_per_token': 0.33,
        'samples_per_second': 10.80,
        'throughput': 10.80,
        'total_params': 7241732096,  # 7.24B
        'active_params': 7241732096,
        'trainable_params': 262410240,  # 262.41M
        'sparsity_ratio': 1.0,
        'model_size_mb': 7156.51,
        'gpu_memory_allocated_gb': 7.03,
        'gpu_memory_reserved_gb': 7.86,
    }

# 2. Dense with KD - Load from file or use provided values
dense_kd = {}
if os.path.exists("results/dense_kd_comprehensive.json"):
    try:
        with open("results/dense_kd_comprehensive.json", 'r') as f:
            dense_kd = json.load(f)
        print("Loaded Dense with KD from results/dense_kd_comprehensive.json")
    except:
        pass

# If file doesn't exist, mark as not available
if not dense_kd or 'accuracy' not in dense_kd:
    print("Dense with KD not yet evaluated - will show N/A for missing metrics")
    dense_kd = {
        'accuracy': None,
        'top2_accuracy': None,
        'ece': None,
        'perplexity': None,
        'flops': None,
        'tokens_per_second': None,
        'ms_per_token': None,
        'samples_per_second': None,
        'throughput': None,
        'total_params': None,
        'active_params': None,
        'trainable_params': None,
        'sparsity_ratio': None,
        'model_size_mb': None,
        'gpu_memory_allocated_gb': None,
        'gpu_memory_reserved_gb': None,
    }

# 3. MoE Standard Training - Load from file or use provided values
moe_standard = {}
if os.path.exists("results/trained_moe_comprehensive.json"):
    try:
        with open("results/trained_moe_comprehensive.json", 'r') as f:
            moe_standard = json.load(f)
        print("Loaded MoE Standard from results/trained_moe_comprehensive.json")
    except:
        pass

# If file doesn't exist or is incomplete, use provided values
if not moe_standard or 'accuracy' not in moe_standard:
    print("Using provided MoE Standard Training values")
    moe_standard = {
        'accuracy': 0.6860,
        'top2_accuracy': 0.8560,
        'ece': 0.0441,
        'perplexity': 4.76,
        'flops': 3023656976384,  # 3023.66G
        'tokens_per_second': 5465.73,
        'ms_per_token': 0.18,
        'samples_per_second': 19.22,
        'throughput': 19.22,
        'total_params': 46716424448,  # 46.72B
        'active_params': 46716424448,
        'trainable_params': 13631488,  # 13.63M
        'sparsity_ratio': 1.0,
        'model_size_mb': 89130.51,
        'gpu_memory_allocated_gb': 95.08,
        'gpu_memory_reserved_gb': 95.56,
    }

# 4. MoE with KD - Use provided values (from training output)
moe_kd = {
    'accuracy': 0.6610,
    'top2_accuracy': 0.8360,
    'ece': 0.0763,
    'perplexity': 6.00,
    'flops': None,  # Not provided in evaluation output
    'tokens_per_second': None,  # Not provided
    'ms_per_token': 0.0543, 
    'samples_per_second': 18.74,
    'throughput': 18.74,
    'total_params': 46716424448,  # Assume same architecture as MoE Standard
    'active_params': 46716424448,
    'trainable_params': 13631488,
    'sparsity_ratio': 1.0,
    'model_size_mb': 89130.51,  # Assume same as MoE Standard
    'gpu_memory_allocated_gb': 97.29,
    'gpu_memory_reserved_gb': None,
}

print("\n" + "=" * 80)
print("RESULTS LOADED")
print("=" * 80)

# Create unified comparison data
unified_data = {
    'Dense Baseline (Standard)': dense_baseline,
    'Dense with KD': dense_kd,
    'MoE Standard Training': moe_standard,
    'MoE Knowledge Distillation': moe_kd,
}

# Extract key metrics for table
table_data = []

for model_name, metrics in unified_data.items():
    # Get or compute metrics
    accuracy = metrics.get('accuracy', None)
    top2_accuracy = metrics.get('top2_accuracy', None)
    
    # Perplexity
    perplexity = metrics.get('perplexity', None)
    if perplexity is None and 'eval_loss' in metrics:
        try:
            perplexity = np.exp(metrics['eval_loss'])
        except:
            perplexity = None
    
    # ECE
    ece = metrics.get('ece', None)
    
    # Throughput (samples/second)
    throughput = metrics.get('throughput', None) or metrics.get('samples_per_second', None)
    
    # Latency (ms/token or avg_latency)
    latency = metrics.get('ms_per_token', None)
    if latency is None and 'avg_latency' in metrics:
        latency = metrics['avg_latency'] * 1000  # Convert to ms
    
    # Tokens per second
    tokens_per_second = metrics.get('tokens_per_second', None)
    
    # Parameters (total)
    params = metrics.get('total_params', None)
    
    # FLOPs
    flops = metrics.get('flops', None)
    
    # GPU Memory
    gpu_memory = metrics.get('gpu_memory_allocated_gb', None)
    
    table_data.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Top-2 Accuracy': top2_accuracy,
        'Perplexity': perplexity,
        'ECE': ece,
        'Throughput (samples/s)': throughput,
        'Latency (ms/token)': latency,
        'Tokens/sec': tokens_per_second,
        'FLOPs (G)': flops / 1e9 if flops is not None else None,
        'Parameters (B)': params / 1e9 if params is not None else None,
        'GPU Memory (GB)': gpu_memory,
    })

# Create DataFrame
df = pd.DataFrame(table_data)

# Display formatted table
print("\n" + "=" * 150)
print("UNIFIED MODEL COMPARISON TABLE")
print("=" * 150)
print()

# Create detailed formatted table
formatted_rows = []
header = f"{'Model':<35} {'Accuracy':<12} {'Top-2':<10} {'Perplexity':<12} {'ECE':<10} {'Throughput':<15} {'Latency':<15} {'Tokens/s':<12} {'FLOPs':<12} {'Params':<12} {'GPU':<12}"
formatted_rows.append(header)
formatted_rows.append("-" * 150)

for _, row in df.iterrows():
    model = row['Model']
    acc = f"{row['Accuracy']:.4f}" if pd.notna(row['Accuracy']) else "N/A"
    top2 = f"{row['Top-2 Accuracy']:.4f}" if pd.notna(row['Top-2 Accuracy']) else "N/A"
    perp = f"{row['Perplexity']:.2f}" if pd.notna(row['Perplexity']) else "N/A"
    ece = f"{row['ECE']:.4f}" if pd.notna(row['ECE']) else "N/A"
    thr = f"{row['Throughput (samples/s)']:.2f}" if pd.notna(row['Throughput (samples/s)']) else "N/A"
    lat = f"{row['Latency (ms/token)']:.2f}" if pd.notna(row['Latency (ms/token)']) else "N/A"
    tok = f"{row['Tokens/sec']:.2f}" if pd.notna(row['Tokens/sec']) else "N/A"
    flops = f"{row['FLOPs (G)']:.2f}G" if pd.notna(row['FLOPs (G)']) else "N/A"
    params = f"{row['Parameters (B)']:.2f}B" if pd.notna(row['Parameters (B)']) else "N/A"
    gpu = f"{row['GPU Memory (GB)']:.2f}GB" if pd.notna(row['GPU Memory (GB)']) else "N/A"
    
    formatted_rows.append(
        f"{model:<35} {acc:<12} {top2:<10} {perp:<12} {ece:<10} {thr:<15} {lat:<15} {tok:<12} {flops:<12} {params:<12} {gpu:<12}"
    )

for row in formatted_rows:
    print(row)

print("=" * 150)
print()

# Print detailed DataFrame
print("Detailed DataFrame:")
print(df.to_string(index=False))
print()

# Save to files
os.makedirs("results", exist_ok=True)

# Save as CSV
csv_path = "results/unified_model_comparison.csv"
df.to_csv(csv_path, index=False)
print(f"Unified comparison table saved to: {csv_path}")

# Save as JSON with full details
json_data = {}
for model_name, metrics in unified_data.items():
    json_data[model_name] = metrics

json_path = "results/unified_model_comparison.json"
with open(json_path, 'w') as f:
    json.dump(json_data, f, indent=2, default=str)
print(f"Unified comparison data saved to: {json_path}")

# Also save the table data
table_json_path = "results/unified_model_comparison_table.json"
with open(table_json_path, 'w') as f:
    json.dump(table_data, f, indent=2, default=str)
print(f"Unified table data saved to: {table_json_path}")


In [ ]:
# ============================================================================
# UPDATED VISUAL COMPARISON - Unified Model Metrics
# ============================================================================

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    # Set style
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (18, 12)
    plt.rcParams['font.size'] = 10
    
    # Prepare data for visualization
    models = df['Model'].values
    metrics_to_plot = {
        'Accuracy': df['Accuracy'].values,
        'ECE': df['ECE'].values,
        'Throughput (samples/s)': df['Throughput (samples/s)'].values,
        'Latency (ms/token)': df['Latency (ms/token)'].values,
        'Parameters (B)': df['Parameters (B)'].values,
    }
    
    # Create subplots - 3 rows, 2 columns for 5 plots
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    fig.suptitle('Unified Model Comparison: Comprehensive Metrics', fontsize=18, fontweight='bold', y=0.995)
    
    # Flatten axes for easier indexing
    axes_flat = axes.flatten()
    
    # Color palette for consistency
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    # Plot 1: Accuracy Comparison
    ax1 = axes_flat[0]
    valid_idx = [i for i, v in enumerate(metrics_to_plot['Accuracy']) if v is not None]
    if valid_idx:
        bars1 = ax1.bar([models[i] for i in valid_idx], 
                [metrics_to_plot['Accuracy'][i] for i in valid_idx],
                color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
        ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
        ax1.set_title('Accuracy Comparison', fontsize=13, fontweight='bold', pad=10)
        ax1.set_ylim([0, max(metrics_to_plot['Accuracy'][i] for i in valid_idx) * 1.15])
        ax1.grid(axis='y', alpha=0.3, linestyle='--')
        ax1.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
        # Add value labels on bars
        for i, (bar, idx) in enumerate(zip(bars1, valid_idx)):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Plot 2: Calibration Error (ECE) Comparison
    ax2 = axes_flat[1]
    valid_idx = [i for i, v in enumerate(metrics_to_plot['ECE']) if v is not None]
    if valid_idx:
        bars2 = ax2.bar([models[i] for i in valid_idx], 
                [metrics_to_plot['ECE'][i] for i in valid_idx],
                color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
        ax2.set_ylabel('ECE (Expected Calibration Error)', fontsize=12, fontweight='bold')
        ax2.set_title('Calibration Error Comparison', fontsize=13, fontweight='bold', pad=10)
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        ax2.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
        # Add value labels on bars
        for i, (bar, idx) in enumerate(zip(bars2, valid_idx)):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        # Lower is better for ECE, so add a note
        ax2.text(0.02, 0.98, 'Lower is Better', transform=ax2.transAxes,
                fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
                facecolor='wheat', alpha=0.5))
    
    # Plot 3: Throughput Comparison
    ax3 = axes_flat[2]
    valid_idx = [i for i, v in enumerate(metrics_to_plot['Throughput (samples/s)']) if v is not None]
    if valid_idx:
        bars3 = ax3.bar([models[i] for i in valid_idx], 
                [metrics_to_plot['Throughput (samples/s)'][i] for i in valid_idx],
                color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
        ax3.set_ylabel('Throughput (samples/sec)', fontsize=12, fontweight='bold')
        ax3.set_title('Throughput Comparison', fontsize=13, fontweight='bold', pad=10)
        ax3.grid(axis='y', alpha=0.3, linestyle='--')
        ax3.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
        # Add value labels on bars
        for i, (bar, idx) in enumerate(zip(bars3, valid_idx)):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        # Higher is better for throughput
        ax3.text(0.02, 0.98, 'Higher is Better', transform=ax3.transAxes,
                fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
                facecolor='lightgreen', alpha=0.5))
    
    # Plot 4: Latency Comparison
    ax4 = axes_flat[3]
    valid_idx = [i for i, v in enumerate(metrics_to_plot['Latency (ms/token)']) if v is not None]
    if valid_idx:
        bars4 = ax4.bar([models[i] for i in valid_idx], 
                [metrics_to_plot['Latency (ms/token)'][i] for i in valid_idx],
                color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
        ax4.set_ylabel('Latency (ms/token)', fontsize=12, fontweight='bold')
        ax4.set_title('Latency Comparison', fontsize=13, fontweight='bold', pad=10)
        ax4.grid(axis='y', alpha=0.3, linestyle='--')
        ax4.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
        # Add value labels on bars
        for i, (bar, idx) in enumerate(zip(bars4, valid_idx)):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        # Lower is better for latency
        ax4.text(0.02, 0.98, 'Lower is Better', transform=ax4.transAxes,
                fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
                facecolor='wheat', alpha=0.5))
    
    # Plot 5: Parameters Comparison
    ax5 = axes_flat[4]
    valid_idx = [i for i, v in enumerate(metrics_to_plot['Parameters (B)']) if v is not None]
    if valid_idx:
        bars5 = ax5.bar([models[i] for i in valid_idx], 
                [metrics_to_plot['Parameters (B)'][i] for i in valid_idx],
                color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
        ax5.set_ylabel('Parameters (Billions)', fontsize=12, fontweight='bold')
        ax5.set_title('Model Size Comparison', fontsize=13, fontweight='bold', pad=10)
        ax5.grid(axis='y', alpha=0.3, linestyle='--')
        ax5.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
        # Add value labels on bars
        for i, (bar, idx) in enumerate(zip(bars5, valid_idx)):
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}B',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Hide the 6th subplot (unused)
    axes_flat[5].axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    
    # Save figure
    fig_path = "results/unified_model_comparison_visualization.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Visualization saved to: {fig_path}")
    
    plt.show()
    
except ImportError:
    print("Matplotlib/Seaborn not available. Skipping visualization.")
except Exception as e:
    print(f"Could not create visualization: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Tokenize the dataset for train/validation splits
# IMPORTANT: Training uses prompts WITH answers, evaluation uses prompts WITHOUT answers

def tokenize_function_for_training(examples):
    """Tokenize WITH answers for training - model learns to predict correct answer."""
    return tokenizer(
        examples['formatted_prompt_with_answer'],  # <-- WITH ANSWER
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

def tokenize_function_for_eval(examples):
    """Tokenize WITHOUT answers for evaluation loss computation."""
    return tokenizer(
        examples['formatted_prompt'],  # <-- WITHOUT ANSWER
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

print("Tokenizing dataset...")
print("  Training data: using prompts WITH answers (teaches model correct answers)")
print("  Eval data: using prompts WITHOUT answers (for loss computation)")

# Tokenize training data WITH answers
tokenized_train = dataset['train'].map(
    tokenize_function_for_training,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing training data (with answers)"
)

# Tokenize eval data WITHOUT answers (for loss computation during training)
tokenized_eval = dataset['test'].map(
    tokenize_function_for_eval,
    batched=True,
    remove_columns=dataset['test'].column_names,
    desc="Tokenizing eval data (without answers)"
)

print("Tokenization complete.")
print(f"  Training samples: {len(tokenized_train):,}")
print(f"  Eval samples: {len(tokenized_eval):,}")

In [ ]:
if USE_SUBSET:
    num_train_samples = int(len(tokenized_train) * SUBSET_PERCENTAGE)
    num_val_samples = int(len(tokenized_eval) * SUBSET_PERCENTAGE)
    
    train_dataset = tokenized_train.select(range(num_train_samples))
    val_dataset = tokenized_eval.select(range(num_val_samples))
    
    print(f"\nUsing {SUBSET_PERCENTAGE*100}% subset of data:")
    print(f"  Train samples: {len(train_dataset)} (with answers)")
    print(f"  Validation samples: {len(val_dataset)} (without answers)")
else:
    train_dataset = tokenized_train
    val_dataset = tokenized_eval
    print(f"\nUsing full dataset:")
    print(f"  Train samples: {len(train_dataset)} (with answers)")
    print(f"  Validation samples: {len(val_dataset)} (without answers)")

In [ ]:
# --------------------------------------------------------------------------
# 8. BUILD TARGET MODULES LIST FOR LORA (INCLUDING ROUTERS!)
# --------------------------------------------------------------------------

print("\nBuilding target modules list for LoRA...")

# Standard attention/MLP modules
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
]



In [ ]:
# --------------------------------------------------------------------------
# 9. APPLY LORA TO STUDENT MODEL
# --------------------------------------------------------------------------

from peft import get_peft_model, LoraConfig, TaskType

print("\nApplying LoRA to student model...")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_CONFIG['r'],
    lora_alpha=LORA_CONFIG['lora_alpha'],
    lora_dropout=LORA_CONFIG['lora_dropout'],
    target_modules=target_modules,
    bias="none",
)

student_model = get_peft_model(student_model, peft_config)

print("LoRA applied successfully!")
print(f"  LoRA rank (r): {LORA_CONFIG['r']}")
print(f"  LoRA alpha: {LORA_CONFIG['lora_alpha']}")
print(f"  LoRA dropout: {LORA_CONFIG['lora_dropout']}")


In [ ]:
# --------------------------------------------------------------------------
# 10. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------------------------------

print("\nTrainable parameter summary:")
student_model.print_trainable_parameters()

# Detailed breakdown
total_params = sum(p.numel() for p in student_model.parameters())
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nDetailed parameter breakdown:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  Frozen parameters: {frozen_params:,} ({100 * frozen_params / total_params:.2f}%)")


In [ ]:
# --------------------------------------------------------------------------
# 11. SETUP TRAINING ARGUMENTS
# --------------------------------------------------------------------------

print("\nSetting up training arguments...")

output_dir = "./moe-mistral-mmlu-lora"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    max_steps=TRAINING_CONFIG['max_steps'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    logging_dir=f"{output_dir}/logs",
    logging_steps=TRAINING_CONFIG['logging_steps'],
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG['eval_steps'],
    save_strategy="steps",
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to=["tensorboard"],
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

print("Training arguments configured:")
print(f"  Output directory: {output_dir}")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Warmup ratio: {TRAINING_CONFIG['warmup_ratio']}")


In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("\nInitializing trainer...")

# Prepare eval datasets for callback
# eval_dataset_for_accuracy needs the RAW dataset with 'formatted_prompt' and 'answer' columns
# eval_tokenized_for_loss needs the TOKENIZED dataset with 'input_ids' for loss computation
eval_dataset_for_accuracy = eval_dataset  # Raw dataset (has 'formatted_prompt', 'answer')
eval_tokenized_for_loss = val_dataset     # Tokenized dataset (has 'input_ids', 'attention_mask')

# Select trainer based on configuration
if USE_KNOWLEDGE_DISTILLATION:
    print("  Using MoEKDTrainer for knowledge distillation")
    trainer = IntegratedMoEKDTrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        teacher_model=teacher_model,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        kd_alpha=KD_CONFIG['kd_alpha'],
        temperature=KD_CONFIG['temperature'],
        routing_kd_weight=KD_CONFIG['routing_kd_weight'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )
else:
    print("  Using MoETrainer for standard training")
    trainer = MoETrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )

print("Trainer initialized successfully!")

In [ ]:
# --------------------------------------------------------------------------
# TRAIN THE MODEL
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)

# Clear GPU cache before training
import gc
torch.cuda.empty_cache()
gc.collect()

print(f"\nTraining configuration:")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Eval steps: {TRAINING_CONFIG['eval_steps']}")

# Start training
train_result = trainer.train()

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

# Print final training metrics
print("\nFinal training metrics:")
for key, value in train_result.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save the final model
print(f"\nSaving model to {output_dir}/final_model...")
trainer.save_model(f"{output_dir}/final_model")
tokenizer.save_pretrained(f"{output_dir}/final_model")
print("Model saved successfully!")

# Trained Model evaluation


In [ ]:
# ============================================================================
# TRAINED MoE MODEL COMPREHENSIVE EVALUATION
# ============================================================================

import json
import os
import numpy as np

print("=" * 80)
print("TRAINED MoE MODEL COMPREHENSIVE EVALUATION")
print("=" * 80)

# --------------------------------------------------------------------------
# 1. SETUP - Load trained model
# --------------------------------------------------------------------------

print("\n📦 Loading trained model...")

# Determine which model to evaluate
if 'student_model' in dir() and student_model is not None:
    trained_model = student_model
    model_name = "Trained MoE Model (in-memory)"
    print(" Using student_model from training session")
else:
    # Load from checkpoint if not in memory
    from peft import PeftModel
    
    # Try to load from the most recent checkpoint
    checkpoint_dir = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "./mistral_moe_qlora"
    
    print(f"Loading from checkpoint: {checkpoint_dir}/final_model")
    
    # Load base MoE model first
    trained_model = moe_model  # Assumes moe_model is still in memory
    
    # Load LoRA weights
    trained_model = PeftModel.from_pretrained(trained_model, f"{checkpoint_dir}/final_model")
    model_name = f"Trained Model from {checkpoint_dir}"
    print(f" Loaded checkpoint: {checkpoint_dir}/final_model")

# Set to eval mode
trained_model.eval()
print(f" Model set to evaluation mode")

# --------------------------------------------------------------------------
# 2. MMLU Accuracy
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. MMLU ACCURACY EVALUATION")
print("=" * 80)

trained_results = evaluate_mmlu_comprehensive(
    model=trained_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

print(f" MMLU Accuracy: {trained_results['accuracy']:.4f}")
print(f" Top-2 Accuracy: {trained_results['top2_accuracy']:.4f}")
print(f" ECE: {trained_results['ece']:.4f}")

# --------------------------------------------------------------------------
# 3. FLOPs Estimation
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. COMPUTATIONAL EFFICIENCY (FLOPs)")
print("=" * 80)

print("Computing FLOPs...")
trained_flops = compute_model_flops(trained_model, seq_length=MAX_LENGTH)
print(f" FLOPs per forward pass: {trained_flops/1e9:.2f}G")

# --------------------------------------------------------------------------
# 4. Throughput Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. THROUGHPUT METRICS")
print("=" * 80)

print("Measuring throughput...")
trained_throughput = compute_throughput_metrics(
    model=trained_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

print(f" Tokens/second: {trained_throughput['tokens_per_second']:.2f}")
print(f" ms/token: {trained_throughput['ms_per_token']:.2f}")
print(f" Samples/second: {trained_throughput['samples_per_second']:.2f}")

# --------------------------------------------------------------------------
# 5. Parameter Efficiency
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. PARAMETER EFFICIENCY")
print("=" * 80)

print("Analyzing parameter efficiency...")

# Determine num_experts_per_tok from model
base_model_ref = trained_model
if hasattr(trained_model, 'base_model'):
    base_model_ref = trained_model.base_model
if hasattr(base_model_ref, 'model'):
    base_model_ref = base_model_ref.model

# Get num_experts_per_tok from first MoE layer
num_experts_per_tok_value = 2  # Default
if hasattr(base_model_ref, 'layers') and len(base_model_ref.layers) > 0:
    first_layer = base_model_ref.layers[0]
    if hasattr(first_layer, 'mlp') and hasattr(first_layer.mlp, 'num_experts_per_tok'):
        num_experts_per_tok_value = first_layer.mlp.num_experts_per_tok

trained_params = compute_parameter_efficiency(
    model=trained_model,
    num_experts_per_tok=num_experts_per_tok_value
)

print(f" Total parameters: {trained_params['total_params']/1e9:.2f}B")
print(f" Active parameters: {trained_params['active_params']/1e9:.2f}B")
print(f" Trainable parameters: {trained_params['trainable_params']/1e6:.2f}M")
print(f" Sparsity ratio: {trained_params['sparsity_ratio']:.2%}")

# --------------------------------------------------------------------------
# 6. Memory Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. MEMORY USAGE")
print("=" * 80)

print("Collecting memory metrics...")
trained_memory = compute_memory_metrics(trained_model)

print(f" Model size: {trained_memory['model_size_mb']:.2f} MB")
print(f" GPU allocated: {trained_memory['gpu_memory_allocated_gb']:.2f} GB")
print(f" GPU reserved: {trained_memory['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 7. Combine All Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINING ALL METRICS")
print("=" * 80)

trained_comprehensive = {
    **trained_results,
    'flops': trained_flops,
    **trained_throughput,
    **trained_params,
    **trained_memory,
}

# --------------------------------------------------------------------------
# 8. Display Comprehensive Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPREHENSIVE TRAINED MoE METRICS")
print("=" * 80)

print("\n Accuracy Metrics:")
print(f"  MMLU Accuracy: {trained_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {trained_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {trained_comprehensive['ece']:.4f}")

print("\n Computational Efficiency:")
print(f"  FLOPs per forward pass: {trained_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {trained_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {trained_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {trained_comprehensive['samples_per_second']:.2f}")

print("\n Parameter Efficiency:")
print(f"  Total parameters: {trained_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {trained_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {trained_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {trained_comprehensive['sparsity_ratio']:.2%}")

print("\n Memory Usage:")
print(f"  Model size: {trained_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {trained_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {trained_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 9. Compare with Baseline (if available)
# --------------------------------------------------------------------------

if 'baseline_comprehensive' in dir():
    print("\n" + "=" * 80)
    print("COMPARISON WITH BASELINE")
    print("=" * 80)
    
    print("\n Accuracy Comparison:")
    acc_diff = trained_comprehensive['accuracy'] - baseline_comprehensive['accuracy']
    acc_change = (acc_diff / baseline_comprehensive['accuracy']) * 100
    print(f"  Baseline Accuracy:     {baseline_comprehensive['accuracy']:.4f}")
    print(f"  Trained Accuracy:      {trained_comprehensive['accuracy']:.4f}")
    print(f"  Difference:            {acc_diff:+.4f} ({acc_change:+.2f}%)")
    
    print("\n⚡ Efficiency Comparison:")
    throughput_diff = trained_comprehensive['tokens_per_second'] - baseline_comprehensive['tokens_per_second']
    throughput_change = (throughput_diff / baseline_comprehensive['tokens_per_second']) * 100
    print(f"  Baseline Tokens/sec:   {baseline_comprehensive['tokens_per_second']:.2f}")
    print(f"  Trained Tokens/sec:    {trained_comprehensive['tokens_per_second']:.2f}")
    print(f"  Difference:            {throughput_diff:+.2f} ({throughput_change:+.2f}%)")
    
    print("\n Memory Comparison:")
    mem_diff = trained_comprehensive['gpu_memory_allocated_gb'] - baseline_comprehensive['gpu_memory_allocated_gb']
    print(f"  Baseline GPU Memory:   {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  Trained GPU Memory:    {trained_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  Difference:            {mem_diff:+.2f} GB")
    
    print("\n Parameter Efficiency:")
    active_ratio_baseline = baseline_comprehensive['active_params'] / baseline_comprehensive['total_params']
    active_ratio_trained = trained_comprehensive['active_params'] / trained_comprehensive['total_params']
    print(f"  Baseline Active/Total: {active_ratio_baseline:.2%}")
    print(f"  Trained Active/Total:  {active_ratio_trained:.2%}")

# --------------------------------------------------------------------------
# 10. Save Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

os.makedirs("results", exist_ok=True)

# Save trained model results
results_file = "results/trained_moe_comprehensive.json"
with open(results_file, 'w') as f:
    json.dump({k: v for k, v in trained_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)
print(f" Saved trained model results to: {results_file}")

# Save comparison if baseline exists
if 'baseline_comprehensive' in dir():
    comparison_results = {
        'baseline': {k: v for k, v in baseline_comprehensive.items() if not isinstance(v, np.ndarray)},
        'trained': {k: v for k, v in trained_comprehensive.items() if not isinstance(v, np.ndarray)},
        'improvements': {
            'accuracy_diff': acc_diff,
            'accuracy_change_pct': acc_change,
            'throughput_diff': throughput_diff,
            'throughput_change_pct': throughput_change,
            'memory_diff_gb': mem_diff,
        }
    }
    
    comparison_file = "results/baseline_vs_trained_comparison.json"
    with open(comparison_file, 'w') as f:
        json.dump(comparison_results, f, indent=2, default=str)
    print(f" Saved comparison results to: {comparison_file}")

# --------------------------------------------------------------------------
# 11. Log to WandB (if available)
# --------------------------------------------------------------------------

try:
    if 'wandb' in dir() and wandb.run is not None:
        print("\n📊 Logging to WandB...")
        wandb.log({
            'trained/accuracy': trained_comprehensive['accuracy'],
            'trained/top2_accuracy': trained_comprehensive['top2_accuracy'],
            'trained/ece': trained_comprehensive['ece'],
            'trained/flops_billions': trained_comprehensive['flops']/1e9,
            'trained/tokens_per_second': trained_comprehensive['tokens_per_second'],
            'trained/ms_per_token': trained_comprehensive['ms_per_token'],
            'trained/active_params_billions': trained_comprehensive['active_params']/1e9,
            'trained/trainable_params_millions': trained_comprehensive['trainable_params']/1e6,
            'trained/gpu_memory_gb': trained_comprehensive['gpu_memory_allocated_gb'],
            'trained/sparsity_ratio': trained_comprehensive['sparsity_ratio'],
        })
        
        # Log comparison if baseline exists
        if 'baseline_comprehensive' in dir():
            wandb.log({
                'comparison/accuracy_improvement': acc_diff,
                'comparison/accuracy_improvement_pct': acc_change,
                'comparison/throughput_improvement': throughput_diff,
                'comparison/throughput_improvement_pct': throughput_change,
            })
        
        print(" Logged to WandB")
except Exception as e:
    print(f"  Could not log to WandB: {e}")

# --------------------------------------------------------------------------
# 12. Store for later use
# --------------------------------------------------------------------------

trained_accuracy = trained_comprehensive['accuracy']

print("\n" + "=" * 80)
print(" COMPREHENSIVE TRAINED MoE EVALUATION COMPLETE!")
print("=" * 80)

# Clean up
torch.cuda.empty_cache()

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("\nInitializing trainer...")

# Prepare eval datasets for callback
# eval_dataset_for_accuracy needs the RAW dataset with 'formatted_prompt' and 'answer' columns
# eval_tokenized_for_loss needs the TOKENIZED dataset with 'input_ids' for loss computation
eval_dataset_for_accuracy = eval_dataset  # Raw dataset (has 'formatted_prompt', 'answer')
eval_tokenized_for_loss = val_dataset     # Tokenized dataset (has 'input_ids', 'attention_mask')

# Select trainer based on configuration
if USE_KNOWLEDGE_DISTILLATION:
    print("  Using MoEKDTrainer for knowledge distillation")
    # CORRECT - passes KD_CONFIG as a dictionary
    trainer = IntegratedMoEKDTrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG,                       # ✅ Pass entire dict
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )
else:
    print("  Using MoETrainer for standard training")
    trainer = MoETrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )

print("Trainer initialized successfully!")

In [ ]:
# --------------------------------------------------------------------------
# TRAIN THE MODEL
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)

# Clear GPU cache before training
import gc
torch.cuda.empty_cache()
gc.collect()

print(f"\nTraining configuration:")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Eval steps: {TRAINING_CONFIG['eval_steps']}")

# Start training
train_result = trainer.train()

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

# Print final training metrics
print("\nFinal training metrics:")
for key, value in train_result.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save the final model
print(f"\nSaving model to {output_dir}/final_model...")
trainer.save_model(f"{output_dir}/final_model")
tokenizer.save_pretrained(f"{output_dir}/final_model")
print("Model saved successfully!")

# MoE routing variants

In [ ]:
from dataclasses import dataclass
from typing import Literal, List, Optional
import json

@dataclass
class MoEExperimentConfig:
    """Configuration for MoE variant experiments."""

    # Basic architecture
    num_experts: int = 8
    num_experts_per_tok: int = 2

    # Routing configuration
    router_jitter_noise: float = 0.0
    router_aux_loss_coef: float = 0.001

    # Expert placement
    expert_layers: Literal["all", "every_2", "every_4", "selected"] = "all"
    layer_indices: Optional[List[int]] = None

    # Training
    load_balancing_loss_coef: float = 0.01

    # Metadata
    experiment_name: str = "default"
    description: str = ""

    def to_dict(self):
        return {
            'num_experts': self.num_experts,
            'num_experts_per_tok': self.num_experts_per_tok,
            'router_jitter_noise': self.router_jitter_noise,
            'router_aux_loss_coef': self.router_aux_loss_coef,
            'expert_layers': self.expert_layers,
            'layer_indices': self.layer_indices,
            'load_balancing_loss_coef': self.load_balancing_loss_coef,
            'experiment_name': self.experiment_name,
            'description': self.description,
        }

    def save(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'r') as f:
            return cls(**json.load(f))


# Predefined experiment configurations
EXPERIMENT_CONFIGS = {
    # --- 1. Routing Variants ---
    "top1_8x1": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_8x1",
        description="Top-1 routing: 8 experts, single expert per token"
    ),

    "top1_16x1": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_16x1",
        description="Top-1 routing: 16 experts, single expert per token"
    ),

    "routing_noisy_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_jitter_noise=0.2,
        expert_layers="all",
        experiment_name="routing_noisy_8x2",
        description="Noisy routing: 8 experts, top-2, high jitter (0.2)"
    ),

    "balanced_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_aux_loss_coef=0.05,  # Higher coefficient for strict balancing
        expert_layers="all",
        experiment_name="balanced_8x2",
        description="Load Balanced: 8 experts, top-2, high aux loss coef (0.05)"
    ),

    # --- 2. Expert Count Variants ---
    "efficient_4x1": MoEExperimentConfig(
        num_experts=4,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="efficient_4x1",
        description="Efficient: 4 experts, top-1 routing"
    ),

    "large_16x2": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=2,
        expert_layers="all",
        experiment_name="large_16x2",
        description="Large: 16 experts, top-2 routing"
    ),

    # --- 3. Placement Variants ---
    "sparse_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="every_2",
        experiment_name="sparse_8x2",
        description="Sparse placement: experts every 2nd layer"
    ),

    "placement_early_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(0, 16)),  # First 16 layers
        experiment_name="placement_early_8x2",
        description="Early placement: Experts in first 16 layers only"
    ),

    "placement_middle_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(8, 24)),  # Middle 16 layers
        experiment_name="placement_middle_8x2",
        description="Middle placement: Experts in middle 16 layers (8-23)"
    ),

    "placement_late_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(16, 32)),  # Last 16 layers
        experiment_name="placement_late_8x2",
        description="Late placement: Experts in last 16 layers only"
    ),

    "placement_mixed_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=[0, 1, 2, 3, 14, 15, 16, 17, 28, 29, 30, 31],  # First 4, Middle 4, Last 4
        experiment_name="placement_mixed_8x2",
        description="Mixed placement: Experts in first 4, middle 4, and last 4 layers"
    ),
}

print(" MoE variant configuration system defined")
print(f"  Available configs: {list(EXPERIMENT_CONFIGS.keys())}")